<div align="center">

# 🌿 Palm SimCLR-YOLO
## Contrastive Self-Supervised Learning for Palm Health Detection

*Bootstrapping YOLOv12s with SimCLR pretraining — learning rich visual representations from unlabeled palm imagery through contrastive objectives, before fine-tuning for 3-class object detection.*

<br>

[![Python](https://img.shields.io/badge/Python-3.12-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://www.python.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.x-EE4C2C?style=for-the-badge&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![Ultralytics](https://img.shields.io/badge/Ultralytics-YOLOv12-00FFFF?style=for-the-badge&logo=yolo&logoColor=black)](https://ultralytics.com/)
[![SimCLR](https://img.shields.io/badge/Self--Supervised-SimCLR-7C3AED?style=for-the-badge)](https://arxiv.org/abs/2002.05709)
[![Albumentations](https://img.shields.io/badge/Augmentation-Albumentations-FF6900?style=for-the-badge)](https://albumentations.ai/)
[![GPU](https://img.shields.io/badge/GPU-Tesla%20T4-76B900?style=for-the-badge&logo=nvidia&logoColor=white)](https://www.nvidia.com/)
[![Status](https://img.shields.io/badge/Status-Research%20%2F%20In%20Progress-FBBF24?style=for-the-badge)](.)

</div>

---

## 📋 Table of Contents

- [✨ At a Glance](#-at-a-glance)
- [💡 The Idea — Why SimCLR?](#-the-idea--why-simclr)
- [🔁 Pipeline Overview](#-pipeline-overview)
- [🗂️ Dataset Deep-Dive](#%EF%B8%8F-dataset-deep-dive)
- [🎨 Data Augmentation](#-data-augmentation)
- [🧬 SimCLR Pretraining — Under the Hood](#-simclr-pretraining--under-the-hood)
- [🎯 Detection Fine-Tuning](#-detection-fine-tuning)
- [🔬 Evaluation Suite](#-evaluation-suite)
- [🔥 Grad-CAM Interpretability](#-grad-cam-interpretability)
- [📦 Export & Deployment](#-export--deployment)
- [🛡️ Engineering Notes](#%EF%B8%8F-engineering-notes)
- [🧰 Tech Stack](#-tech-stack)
- [▶️ Reproducing This Notebook](#%EF%B8%8F-reproducing-this-notebook)
- [🙏 Acknowledgments](#-acknowledgments)

---

## ✨ At a Glance

<div align="center">

| Attribute | Detail |
|:---|:---|
| 🎯 **Task** | 3-class object detection — `abnormal_palm` · `dead_palm` · `healthy_palm` |
| 🧠 **Backbone** | YOLOv12s, pretrained via **SimCLR** (NT-Xent contrastive loss) |
| 📦 **Dataset** | Roboflow `18i0416-abdullah / palm-apuo6 v3` — COCO format |
| 🖼️ **SSL Image Size** | 320 × 320 (SimCLR views) |
| 🖼️ **Detection Image Size** | 640 × 640 (fine-tuning & evaluation) |
| 🔢 **SSL Batch Size** | 64 (large batch critical for contrastive learning) |
| 📈 **Augmented Train Images** | Original + up to 1,500 extra synthetic samples |
| ⚙️ **Hardware** | Tesla T4, Kaggle environment |
| 🌡️ **NT-Xent Temperature** | 0.2 |
| 🔑 **Seed** | 42 |

</div>

---

## 💡 The Idea — Why SimCLR?

Annotating palm health at box-level is slow and expert-intensive. The raw pixel stream is abundant and free.

**SimCLR (Simple Framework for Contrastive Learning of Visual Representations)** exploits that abundance. Given any image, it generates two strongly augmented views and trains a network to produce *similar* representations for the same image and *dissimilar* ones for different images — using the rest of the batch as implicit negatives.

$$\mathcal{L}_{\text{NT-Xent}} = -\frac{1}{2N}\sum_{k=1}^{2N}\log\frac{\exp\!\left(\text{sim}(z_{i(k)},z_{j(k)})/\tau\right)}{\sum_{l=1}^{2N}\mathbb{1}_{[l\neq k]}\exp\!\left(\text{sim}(z_k,z_l)/\tau\right)}$$

The backbone pre-trained this way captures texture, shape, and color structure — features that matter for discriminating healthy from diseased canopies — before a single label is seen. Fine-tuning on top of these representations requires less data to reach high mAP and generalises better to novel palm appearances.

> Unlike BYOL (which needs a momentum-updated target network), SimCLR is **symmetric and single-network** — simpler to implement correctly, but it demands **large batch sizes** for the implicit negatives to cover the contrast space adequately. This notebook uses batch-64 SSL.

---

## 🔁 Pipeline Overview

```
╔══════════════╗   ╔═══════════════╗   ╔══════════════════════╗   ╔══════════════════╗   ╔═════════════════╗
║  📥 DATA     ║   ║  🎨 AUGMENT    ║   ║  🧬 SIMCLR PRETRAIN  ║   ║  🎯 FINETUNE     ║   ║  📊 EVALUATE     ║
║              ║   ║               ║   ║                      ║   ║                  ║   ║                 ║
║ Roboflow     ║──▶║ Albumentations║──▶║ NT-Xent · 70 epochs  ║──▶║ YOLO · 70 epochs ║──▶║ mAP · κ · CM    ║
║ COCO → YOLO  ║   ║ +1,500 extra  ║   ║ 320px · batch 64     ║   ║ 640px · batch 16 ║   ║ Grad-CAM        ║
║ 3 classes    ║   ║ train only    ║   ║ AdamW · AMP          ║   ║ SimCLR init      ║   ║ Conf curve      ║
╚══════════════╝   ╚═══════════════╝   ╚══════════════════════╝   ╚══════════════════╝   ╚═════════════════╝
                                                                                                    │
                                                                                         ╔══════════╧═════╗
                                                                                         ║  📤 EXPORT     ║
                                                                                         ║ PT · ONNX · PKL║
                                                                                         ╚════════════════╝
```

---

## 🗂️ Dataset Deep-Dive

**Source:** Roboflow `18i0416-abdullah / palm-apuo6`, version 3, COCO format.

### Conversion Notes

The COCO → YOLO conversion uses a streamlined `coco2yolo()` function. Before augmentation, all original label files are sanitized — corners clamped to [0, 1] and degenerate boxes discarded — via `clean_all_labels()`. Categories present in annotation metadata but absent from training instances are detected and reported.

---

## 🎨 Data Augmentation

Augmentation is applied **exclusively to the training split**. Validation and test images are copied through untouched. The augmented dataset (`0_yolo_split_aug`) feeds both the SimCLR pretrainer and the detection fine-tuner.

### Offline Augmentation (Albumentations)

| Transform | Parameters | Probability |
|:---|:---|:---:|
| `HorizontalFlip` | — | 0.5 |
| `VerticalFlip` | — | 0.5 |
| `RandomRotate90` | — | 0.5 |
| `Rotate` | limit=15°, `BORDER_REFLECT_101` | 0.5 |
| `HueSaturationValue` | hue=10, sat=50, val=30 | 0.7 |
| `RandomBrightnessContrast` | ±0.2 | 0.7 |
| `GaussianBlur` | kernel (3, 7) | 0.3 |

> **Target:** up to **1,500 extra** training images (`TARGET_EXTRA_AUG=1500`), 3 variants per selected source (`N_AUG_PER_IMAGE=3`). This is more aggressive than a typical BYOL setup — VerticalFlip and RandomRotate90 are ON by default, and ColorJitter strength is nearly 4× stronger in the SimCLR view transform.

BboxParams: `format="yolo"`, `min_visibility=0.3`. Each augmented bounding box is re-sanitized after transform.


## 🧬 SimCLR Pretraining — Under the Hood

### View Augmentation (`SimCLRTransform`)

Two strongly augmented, **independently sampled** views of each image are generated at 320 × 320. Inputs are kept in `[0, 1]` without ImageNet normalization so they remain compatible with YOLO's expected input range.

| Transform | Parameters | Probability |
|:---|:---|:---:|
| `RandomResizedCrop` | size=320, scale=(0.2, 1.0) | 1.0 |
| `RandomHorizontalFlip` | — | 0.5 |
| `ColorJitter` | brightness=0.8, contrast=0.8, saturation=0.8, hue=0.2 | 0.8 |
| `RandomGrayscale` | — | 0.2 |
| `GaussianBlur` | kernel=int(0.1 × 320) | 1.0 |
| `ToTensor` | → [0, 1] | 1.0 |

### Architecture — `YoloSimCLREncoder`

```
  Input [B, 3, 320, 320]
        │
        ▼
  YOLO detection model (forward pass triggers hook)
        │
        └── forward_hook on layer[backbone_idx - 1 before Detect]
              │
              ▼
        [B, C, H, W]  (feature maps from last backbone block)
              │
        Global Average Pool → [B, C]  (feature dim inferred dynamically)
              │
              ▼
        Projection Head g(·):
          Linear(C → 2048) → ReLU → Linear(2048 → 128)
              │
              ▼
        z  [B, 128]  ← L2-normalized before loss
```

**Backbone index discovery:** `find_backbone_idx()` scans the YOLO `ModuleList` for the `Detect` head (importing from `ultralytics.nn.modules.head` with a fallback to `ultralytics.nn.modules`), then returns `detect_idx - 1`. Falls back to `-2` if `Detect` is not found.

**Feature dimension** is inferred dynamically by a dummy `[1, 3, 320, 320]` forward pass — no hardcoding.

### NT-Xent Loss (SimCLR)

```python
def nt_xent_loss(z_i, z_j, temperature=0.2):
    z  = L2_normalize(cat([z_i, z_j]))      # [2N, 128]
    S  = z @ z.T / temperature              # [2N, 2N] cosine sim matrix
    S  = mask_diagonal(S, torch.finfo.min)  # overflow-safe self-mask
    labels = (arange(2N) + N) % 2N          # positives shifted by N
    return cross_entropy(S, labels)
```

Each sample's positive is its augmented twin; all other 2N−2 samples in the batch are negatives. The diagonal mask uses `torch.finfo(sim.dtype).min` rather than `-inf` for float16 safety.

### Hyperparameters

| Parameter | Value |
|:---|:---|
| Epochs | 70 |
| Batch size | **64** (large batch is essential for SimCLR) |
| SSL image size | 320 × 320 |
| DataLoader workers | 4 · `pin_memory=True` · `drop_last=True` |
| Optimizer | AdamW |
| Learning rate | 1e-3 |
| Weight decay | 1e-4 |
| Temperature τ | **0.2** |
| Projection dim | 128 |
| Hidden dim | 2,048 |
| Mixed precision | `torch.amp.GradScaler` → fallback to `torch.cuda.amp` |
| Intermediate checkpoints | Every 10 epochs |
| Final checkpoint | `simclr_backbone_with_hooks_yolov12s.pt` |

> **No EMA, no target network.** SimCLR is a single symmetric network — the same encoder processes both views. This distinguishes it architecturally from BYOL.

---

## 🎯 Detection Fine-Tuning

### Weight Loading

```python
ckpt = torch.load(SIMCLR_SSL_W, map_location="cpu")
simclr_state = ckpt["det_model_state_dict"]
missing, unexpected = model.model.load_state_dict(simclr_state, strict=False)
```

The checkpoint stores both `det_model_state_dict` and `projection_head_state_dict` (kept for reference). Only backbone/neck weights transfer to the detector; the Detect head is randomly re-initialized.

### Training Configuration

| Parameter | Value |
|:---|:---|
| Training data | `data_simclr_augmented.yaml` (augmented train split) |
| Epochs | 70 |
| Image size | 640 × 640 |
| Batch size | 16 |
| Optimizer | (Ultralytics default AdamW) |
| Initial LR `lr0` | 0.005 |
| Weight decay | 6e-4 |
| Early stopping patience | 7 |
| `pretrained` | `False` (SimCLR init, not ImageNet) |
| Run name | `yolo12s_simclr_det` |

---

## 🛡️ Engineering Notes

| Topic | Detail |
|:---|:---|
| **Batch size matters** | SimCLR requires large batches for enough negatives; batch=64 is used for SSL. Fine-tuning uses batch=16 |
| **No target network** | SimCLR is fully symmetric — no EMA, no momentum, no predictor MLP needed |
| **Hook type** | `register_forward_hook` on the backbone output layer (captures activations); unlike BYOL's Detect pre-hook, this captures the final backbone representation directly |
| **Feature dim** | Inferred dynamically via a `[1,3,320,320]` dummy forward pass to avoid hardcoding arch-specific channel counts |
| **Overflow-safe loss** | Diagonal mask uses `torch.finfo(sim.dtype).min` instead of `-inf` for float16 stability |
| **AMP fallback** | `torch.amp.GradScaler` → `torch.cuda.amp.GradScaler` depending on PyTorch version |
| **No eval on SSL data** | Val/test splits are copied without augmentation and never seen during pretraining |
| **Label sanitization** | `clean_all_labels()` clamps all existing label files before augmentation begins |
| **Checkpoint structure** | Each checkpoint (per 10 epochs + final) stores `{det_model_state_dict, projection_head_state_dict}` for reproducibility |
| **Grad-CAM hook cleanup** | `handle_fwd.remove()` and `handle_bwd.remove()` called after each Grad-CAM session |
| **Confidence sweep** | CDF analysis runs at `conf=0.001` to expose the full raw score distribution before thresholding |

---

## 🔗 Related Work

This notebook is part of a broader ablation series comparing self-supervised initialization strategies for palm health detection:

| Notebook | SSL Method | Key Difference |
|:---|:---|:---|
| **This notebook** | SimCLR (NT-Xent) | Symmetric · Large batch · No EMA |
| BYOL-YOLO | BYOL | Asymmetric · EMA target · Small batch |
| Soft Teacher | Semi-supervised | Pseudo-label teacher · Labeled + unlabeled |

---

## 🙏 Acknowledgments

Dataset hosted via **Roboflow** (`18i0416-abdullah / palm-apuo6 v3`). Detector built on **Ultralytics YOLOv12**. Self-supervised pretraining follows the **SimCLR** method (Chen et al., 2020) — *A Simple Framework for Contrastive Learning of Visual Representations* — adapted here for detection backbones rather than image classifiers.

---

<div align="center">

*Made with 🌴 contrastive learning, large batches, and careful augmentation design.*

</div>

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
# ===============================================================
# 0) Setup
# ===============================================================
!pip -q install ultralytics timm
import ultralytics
ultralytics.checks()

import os, cv2, json, yaml, math, gc, random, shutil, contextlib
from pathlib import Path
from PIL import Image

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from collections import Counter
from sklearn.cluster import KMeans
from collections import Counter, defaultdict

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print("Device:", device, "| torch", torch.__version__)

def autocast_ctx():
    return torch.autocast(device_type="cuda", enabled=True) if device=="cuda" else contextlib.nullcontext()

Ultralytics 8.3.235 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6567.2/8062.4 GB disk)
Device: cuda | torch 2.6.0+cu124


In [ ]:
!pip install roboflow
from roboflow import Roboflow
%pip install ultralytics supervision roboflow

rf = Roboflow(api_key="kb3t7HveaF3ISN0TEWpk")
project = rf.workspace("18i0416-abdullah").project("palm-apuo6")
version = project.version(3)
dataset = version.download("coco")

print("Downloaded dataset to:", dataset.location)

In [ ]:
from pathlib import Path
import torch

# Paths (SimCLR version of your previous BYOL notebook)
BASE        = Path("/kaggle/working")

# If you've moved/copied your data under this new folder:
WORK        = BASE / "palm_simclr_ssl"

AUG_SPLIT   = WORK / "0_yolo_split_aug"                 # augmented YOLO split
DATA_AUG    = WORK / "data_simclr_augmented.yaml"       # SimCLR yaml

SIMCLR_SSL_W = WORK / "simclr_backbone_with_hooks_yolov12s.pt"

MODEL_CFG   = "yolov12s.yaml"   

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("AUG_SPLIT:", AUG_SPLIT)
print("DATA_AUG :", DATA_AUG)
print("SIMCLR_SSL_W:", SIMCLR_SSL_W)


# COCO to YOLO conversion¶

In [ ]:
from pathlib import Path
import json, shutil, yaml

# ===============================================================
# 1) Paths (from previous step)
# ===============================================================
BASE        = Path("/kaggle/working")
DATASET_DIR = BASE / "Dat-Palm-Fx-1"        
WORK  = BASE / "palm_simclr_ssl"
SPLIT = WORK / "0_yolo_split"
DATA  = WORK / "data_simclr.yaml"
SSL_W      = WORK / "simclr_backbone_with_hooks_yolov12s.pt"
YOL0V12S_W = WORK / "yolov12s.pt"

WORK.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------
# Utility: COCO bbox -> YOLO (x_center, y_center, w, h) normalized
# ---------------------------------------------------------------
def coco2yolo(b, w, h):
    x, y, bw, bh = b
    return (x + bw/2) / w, (y + bh/2) / h, bw / w, bh / h

# ---------------------------------------------------------------
# Build a canonical mapping (id2new) from the TRAIN annotations:
# only include category ids that actually appear in annotations.
# ---------------------------------------------------------------
train_ann_path = DATASET_DIR / "train" / "_annotations.coco.json"
if not train_ann_path.exists():
    raise FileNotFoundError(f"Train annotation file not found: {train_ann_path}")

train_coco = json.load(open(train_ann_path, "r"))
all_categories = sorted(train_coco.get("categories", []), key=lambda x: x["id"])
present_cat_ids = set(a["category_id"] for a in train_coco.get("annotations", []))

# Filter categories to those present in train annotations
categories = [c for c in all_categories if c["id"] in present_cat_ids]

# Warn about any categories that exist but have zero instances
empty_cats = [c for c in all_categories if c["id"] not in present_cat_ids]
if empty_cats:
    print(" The following categories exist in 'categories' but have zero instances in train annotations:")
    for c in empty_cats:
        print(f"   id={c['id']}\tname={c['name']}")
    print("They will be excluded from the YOLO mapping (nc will decrease).")

# Create mapping: original category_id -> continuous 0..nc-1
id2new = {cat["id"]: i for i, cat in enumerate(categories)}
print("Using category mapping (original id -> new index):")
for orig, new in id2new.items():
    print(f"   {orig} -> {new}")

# ---------------------------------------------------------------
# Conversion function: uses the global id2new mapping above
# ---------------------------------------------------------------
def convert(split_name, img_dir, ann_json_path):
    out_im = SPLIT / split_name / "images"
    out_lb = SPLIT / split_name / "labels"
    out_im.mkdir(parents=True, exist_ok=True)
    out_lb.mkdir(parents=True, exist_ok=True)

    ann_json_path = Path(ann_json_path)
    if not ann_json_path.exists():
        raise FileNotFoundError(f"Annotations file not found: {ann_json_path}")

    coco = json.load(open(ann_json_path, "r"))
    # build image id -> info map
    id2img = {
        im["id"]: {
            k: im[k] for k in ("id", "file_name", "width", "height")
        }
        for im in coco["images"]
    }

    skipped_annotations = 0
    for ann in coco["annotations"]:
        orig_cat = ann["category_id"]
        if orig_cat not in id2new:
            # Skip categories not present in train mapping (or empty categories)
            skipped_annotations += 1
            continue

        img = id2img[ann["image_id"]]
        yb = coco2yolo(ann["bbox"], img["width"], img["height"])
        cls = id2new[orig_cat]

        lbl_path = out_lb / f"{Path(img['file_name']).stem}.txt"
        with open(lbl_path, "a") as f:
            f.write(f"{cls} " + " ".join(f"{v:.6f}" for v in yb) + "\n")

    if skipped_annotations:
        print(f" Skipped {skipped_annotations} annotations because their category_id is not in the train mapping.")

    # copy images
    img_dir = Path(img_dir)
    missing = 0
    for im in coco["images"]:
        src = img_dir / im["file_name"]
        if src.exists():
            shutil.copy(src, out_im / im["file_name"])
        else:
            missing += 1

    if missing:
        print(f" {missing} image files missing from {img_dir} (referenced in annotations but not found).")

# ----------------- run conversion -----------------
# Remove old output to avoid mixing previous runs
if SPLIT.exists():
    shutil.rmtree(SPLIT)
    print("Old YOLO split removed:", SPLIT)

print("➤ Converting COCO → YOLO …")
convert("train", DATASET_DIR / "train", DATASET_DIR / "train" / "_annotations.coco.json")
convert("valid", DATASET_DIR / "valid", DATASET_DIR / "valid" / "_annotations.coco.json")
convert("test",  DATASET_DIR / "test",  DATASET_DIR / "test"  / "_annotations.coco.json")

# write data.yaml (names ordered by our remapping 'categories')
names = [c["name"] for c in categories]
DATA.parent.mkdir(parents=True, exist_ok=True)
DATA.write_text(yaml.dump({
    "path": str(SPLIT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(names),
    "names": names
}, sort_keys=False))

print("✓ YOLO split ready at", SPLIT)
print("classes (nc):", len(names), names)


In [ ]:
train_ann_path = DATASET_DIR / "train" / "_annotations.coco.json"
with open(train_ann_path, "r") as f:
    coco_train = json.load(f)

# category_id -> name
id_to_name = {c["id"]: c["name"] for c in coco_train["categories"]}

# count instances per category_id
cnt = Counter(ann["category_id"] for ann in coco_train["annotations"])

labels = [id_to_name[cid] for cid in cnt.keys()]
values = [cnt[cid] for cid in cnt.keys()]

plt.figure(figsize=(7, 7))
plt.pie(
    values,
    labels=labels,
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False
)
plt.title("Class Distribution (Train COCO)")
plt.axis("equal")  # makes it a perfect circle
plt.tight_layout()
plt.show()


In [ ]:
from pathlib import Path
import math, random
import matplotlib.pyplot as plt
import cv2
import yaml

# ===============================================================
# 1) Paths
# ===============================================================
BASE        = Path("/kaggle/working")
DATASET_DIR = BASE / "Dat-Palm-Fx-1"        
WORK  = BASE / "palm_simclr_ssl"
SPLIT = WORK / "0_yolo_split"
DATA  = WORK / "data_simclr.yaml"
SSL_W      = WORK / "simclr_backbone_with_hooks_yolov12s.pt"
YOL0V12S_W = WORK / "yolov12s.pt"


# ===============================================================
# 2) Load class names from data_byol.yaml
# ===============================================================
with open(DATA, "r") as f:
    data_cfg = yaml.safe_load(f)
class_names = data_cfg["names"]
print("Classes:", class_names)

# ===============================================================
# 3) Helper functions
# ===============================================================
def load_yolo_labels(lbl_path):
    boxes = []
    if not lbl_path.exists():
        return boxes
    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls = int(parts[0])
            x_c, y_c, w, h = map(float, parts[1:])
            boxes.append((cls, x_c, y_c, w, h))
    return boxes

def draw_yolo_boxes(img_path, lbl_path, class_names):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    boxes = load_yolo_labels(lbl_path)
    for cls, x_c, y_c, bw, bh in boxes:
        # YOLO (normalized) -> pixel coords
        x_c_abs = x_c * w
        y_c_abs = y_c * h
        bw_abs = bw * w
        bh_abs = bh * h

        x1 = int(x_c_abs - bw_abs / 2)
        y1 = int(y_c_abs - bh_abs / 2)
        x2 = int(x_c_abs + bw_abs / 2)
        y2 = int(y_c_abs + bh_abs / 2)

        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cls_name = class_names[cls] if cls < len(class_names) else str(cls)
        cv2.putText(img, cls_name, (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1, cv2.LINE_AA)
    return img

# ===============================================================
# 4) Show a grid of random train images with boxes
# ===============================================================
train_img_dir = SPLIT / "train" / "images"
train_lbl_dir = SPLIT / "train" / "labels"

all_imgs = list(train_img_dir.glob("*.jpg")) + list(train_img_dir.glob("*.png"))
random.shuffle(all_imgs)
n_show = min(8, len(all_imgs))

cols = 4
rows = math.ceil(n_show / cols)

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = axes.flatten() if rows * cols > 1 else [axes]

for i in range(rows * cols):
    ax = axes[i]
    ax.axis("off")

    if i >= n_show:
        continue

    img_path = all_imgs[i]
    lbl_path = train_lbl_dir / (img_path.stem + ".txt")
    vis = draw_yolo_boxes(img_path, lbl_path, class_names)

    if vis is None:
        continue

    ax.imshow(vis)
    ax.set_title(f"Img {i+1}", fontsize=10)

fig.suptitle("Sample Training Images with YOLO Annotations", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


# Data Preprocessing

### bad image check (manual pass)

In [ ]:
from pathlib import Path
import random, math
import cv2
import matplotlib.pyplot as plt

from ultralytics import YOLO

# --- paths from your earlier cells ---
BASE        = Path("/kaggle/working")
DATASET_DIR = BASE / "Dat-Palm-Fx-1"        
WORK  = BASE / "palm_simclr_ssl"
SPLIT = WORK / "0_yolo_split"
DATA  = WORK / "data_simclr.yaml"
SSL_W      = WORK / "simclr_backbone_with_hooks_yolov12s.pt"
YOL0V12S_W = WORK / "yolov12s.pt"



def show_random_images(n=16, split="train"):
    img_dir = SPLIT / split / "images"
    imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
    random.shuffle(imgs)
    imgs = imgs[:min(n, len(imgs))]

    cols = 4
    rows = math.ceil(len(imgs) / cols)
    plt.figure(figsize=(4 * cols, 4 * rows))

    for i, p in enumerate(imgs):
        img = cv2.imread(str(p))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(p.name[:20] + "…")

    plt.tight_layout()
    plt.show()

# run this a few times, visually scan for broken / irrelevant images
show_random_images(n=16, split="train")


# Data Augmentation 

In [ ]:
# ===============================================================
# 1) Imports & paths
# ===============================================================
!pip install -q albumentations==1.4.7
from pathlib import Path
import cv2
import shutil
import yaml
from tqdm import tqdm
import albumentations as A
import random
import math
import os
import sys

# ---- Reproducibility
SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

BASE        = Path("/kaggle/working")
DATASET_DIR = BASE / "Dat-Palm-Fx-1"        
WORK        = BASE / "palm_simclr_ssl"
SPLIT       = WORK / "0_yolo_split"              # produced by your COCO->YOLO code
DATA        = WORK / "data_simclr.yaml"          # written by your COCO->YOLO code
SSL_W       = WORK / "simclr_backbone_with_hooks_yolov12s.pt"
YOL0V12S_W  = WORK / "yolov12s.pt"

WORK.mkdir(parents=True, exist_ok=True)

# New augmented dataset root (wipe to avoid mixing old runs)
AUG_SPLIT = WORK / "0_yolo_split_aug"
if AUG_SPLIT.exists():
    shutil.rmtree(AUG_SPLIT)
AUG_SPLIT.mkdir(parents=True, exist_ok=True)

print("Original split:", SPLIT)
print("Augmented split:", AUG_SPLIT)

# Guard: ensure the source split and DATA yaml exist
if not SPLIT.exists():
    raise FileNotFoundError(f"SPLIT not found: {SPLIT} (run your COCO->YOLO conversion first)")
if not DATA.exists():
    raise FileNotFoundError(f"DATA yaml not found: {DATA} (expected from your COCO->YOLO step)")

# Handy: accepted image extensions
IMG_EXTS = (".jpg", ".jpeg", ".png", ".JPG", ".PNG")

# ===============================================================
# 2) Box sanitization helpers
# ===============================================================
def sanitize_box(xc, yc, w, h, eps=1e-6):
    """
    Clamp YOLO box (xc, yc, w, h) so that the *corners* are inside [0,1].
    If the box becomes degenerate (too small), return None.
    """
    if w <= 0 or h <= 0:
        return None

    x1 = xc - w / 2.0
    y1 = yc - h / 2.0
    x2 = xc + w / 2.0
    y2 = yc + h / 2.0

    x1 = max(0.0, min(1.0, x1))
    y1 = max(0.0, min(1.0, y1))
    x2 = max(0.0, min(1.0, x2))
    y2 = max(0.0, min(1.0, y2))

    if x2 - x1 < eps or y2 - y1 < eps:
        return None

    xc = (x1 + x2) / 2.0
    yc = (y1 + y2) / 2.0
    w  = (x2 - x1)
    h  = (y2 - y1)
    return xc, yc, w, h


def clean_label_file(lbl_path: Path):
    """Read a YOLO txt, clamp all boxes to [0,1], rewrite."""
    if not lbl_path.exists():
        return
    new_lines = []
    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            try:
                cls = int(parts[0])
                x_c, y_c, w, h = map(float, parts[1:])
            except Exception:
                continue
            sanitized = sanitize_box(x_c, y_c, w, h)
            if sanitized is None:
                continue
            x_c, y_c, w, h = sanitized
            new_lines.append(f"{cls} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

    with open(lbl_path, "w") as f:
        for ln in new_lines:
            f.write(ln)


def clean_all_labels():
    """Clean labels in train/valid/test of the original SPLIT."""
    for split in ["train", "valid", "test"]:
        lbl_dir = SPLIT / split / "labels"
        if not lbl_dir.exists():
            continue
        print(f"Sanitizing labels in: {lbl_dir}")
        for lbl_path in tqdm(list(lbl_dir.glob("*.txt"))):
            clean_label_file(lbl_path)

clean_all_labels()
print("✓ All original labels sanitized with corner-clamping.")

# ===============================================================
# 3) YOLO label helpers (using sanitized files)
# ===============================================================
def load_yolo_labels(lbl_path: Path):
    """Load YOLO labels from a TXT file. Returns: list[(cls, xc, yc, w, h)]."""
    boxes = []
    if not lbl_path.exists():
        return boxes
    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            try:
                cls = int(parts[0])
                x_c, y_c, w, h = map(float, parts[1:])
            except Exception:
                continue
            sanitized = sanitize_box(x_c, y_c, w, h)
            if sanitized is None:
                continue
            x_c, y_c, w, h = sanitized
            boxes.append((cls, x_c, y_c, w, h))
    return boxes


def save_yolo_labels(lbl_path: Path, boxes):
    """Save list of (cls, x_c, y_c, w, h) to YOLO txt file."""
    with open(lbl_path, "w") as f:
        for cls, x_c, y_c, w, h in boxes:
            f.write(f"{cls} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

# ===============================================================
# 4) Albumentations augmentation pipeline (safe for YOLO bboxes)
# ===============================================================
train_transform = A.Compose(
    [
        # Geometric
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.5),

        # Photometric
        A.HueSaturationValue(
            hue_shift_limit=10,
            sat_shift_limit=50,
            val_shift_limit=30,
            p=0.7,
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.2,
            contrast_limit=0.2,
            p=0.7,
        ),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    ],
    bbox_params=A.BboxParams(
        format="yolo",               # (x_c, y_c, w, h), normalized
        label_fields=["class_labels"],
        min_visibility=0.3,
        # filter_lost_elements=True is default in newer Albumentations;
        # kept implicit to avoid version pinning issues.
    ),
)

# ===============================================================
# 5) Function to augment one split (with max_extra_aug control)
# ===============================================================
def augment_split(split_name, n_aug_per_image=3, max_extra_aug=None):
    """
    split_name: 'train', 'valid', or 'test'
    n_aug_per_image: how many augmented copies to create per selected image
    max_extra_aug: maximum number of extra augmented images (only for train).
                   If None, augment all train images normally.
    """
    src_img_dir = SPLIT / split_name / "images"
    src_lbl_dir = SPLIT / split_name / "labels"

    dst_img_dir = AUG_SPLIT / split_name / "images"
    dst_lbl_dir = AUG_SPLIT / split_name / "labels"
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    img_files = [p for p in src_img_dir.rglob("*") if p.suffix in IMG_EXTS]
    print(f"[{split_name}] Found {len(img_files)} images")

    # Decide which images to augment (only for train)
    if split_name == "train" and max_extra_aug is not None and n_aug_per_image > 0:
        random.shuffle(img_files)
        # how many images need augmentation to reach ~max_extra_aug
        num_imgs_to_augment = math.ceil(max_extra_aug / max(1, n_aug_per_image))
        num_imgs_to_augment = min(num_imgs_to_augment, len(img_files))
        aug_set = set(img_files[:num_imgs_to_augment])
        print(
            f"[{split_name}] Will augment {num_imgs_to_augment} images "
            f"to target ≈{max_extra_aug} extra samples (≈{n_aug_per_image} per image)."
        )
    else:
        aug_set = set(img_files)  # default: all images can be augmented

    extra_created = 0

    for img_path in tqdm(img_files):
        stem = img_path.stem
        lbl_path = src_lbl_dir / f"{stem}.txt"

        # 1) Copy original image + label (always)
        dst_img = dst_img_dir / img_path.name
        dst_lbl = dst_lbl_dir / f"{stem}.txt"
        shutil.copy2(img_path, dst_img)
        if lbl_path.exists():
            shutil.copy2(lbl_path, dst_lbl)

        # Only augment train split
        if split_name != "train" or n_aug_per_image <= 0:
            continue

        # If this image is not in the augment subset, skip augment step
        if img_path not in aug_set:
            continue

        # If we already reached max_extra_aug, stop augmenting
        if max_extra_aug is not None and extra_created >= max_extra_aug:
            continue

        # Load image & labels for augmentation
        img = cv2.imread(str(img_path))
        if img is None:
            # skip unreadable file
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        boxes = load_yolo_labels(lbl_path)
        if not boxes:
            # no labels -> skip augmentation for this file, but original was copied
            continue

        # Albumentations expects: bboxes as [x_c, y_c, w, h], labels separately
        bboxes = [[b[1], b[2], b[3], b[4]] for b in boxes]
        class_labels = [b[0] for b in boxes]

        # 3) Create augmented copies
        for k in range(n_aug_per_image):
            if max_extra_aug is not None and extra_created >= max_extra_aug:
                break

            augmented = train_transform(
                image=img,
                bboxes=bboxes,
                class_labels=class_labels,
            )

            aug_img = augmented["image"]
            aug_bboxes = augmented["bboxes"]
            aug_labels = augmented["class_labels"]

            # if all boxes got dropped, skip this example
            if len(aug_bboxes) == 0:
                continue

            new_stem = f"{stem}_aug{k+1}"
            out_img_path = dst_img_dir / f"{new_stem}.jpg"
            out_lbl_path = dst_lbl_dir / f"{new_stem}.txt"

            # save image
            aug_img_bgr = cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
            cv2.imwrite(str(out_img_path), aug_img_bgr)

            # save labels in YOLO format (sanitize once more, just in case)
            new_boxes = []
            for cls, (xc, yc, bw, bh) in zip(aug_labels, aug_bboxes):
                sanitized = sanitize_box(xc, yc, bw, bh)
                if sanitized is None:
                    continue
                xc, yc, bw, bh = sanitized
                new_boxes.append((cls, xc, yc, bw, bh))
            if new_boxes:
                save_yolo_labels(out_lbl_path, new_boxes)
                extra_created += 1

    total_imgs = len([*dst_img_dir.glob("*.jpg"), *dst_img_dir.glob("*.jpeg"), *dst_img_dir.glob("*.png"),
                      *dst_img_dir.glob("*.JPG"), *dst_img_dir.glob("*.PNG")])
    print(f"[{split_name}] Done. Images in augmented split: {total_imgs}")
    if split_name == "train":
        print(f"[{split_name}] Extra augmented images created: {extra_created}")

# ===============================================================
# 6) Run augmentation
# ===============================================================
# Ei duita parameter diye tumi control korte parba koto extra image hobe
TARGET_EXTRA_AUG = 1500   # e.g. 1000–2000 extra train images
N_AUG_PER_IMAGE  = 3      # how many variants per selected image

augment_split("train", n_aug_per_image=N_AUG_PER_IMAGE, max_extra_aug=TARGET_EXTRA_AUG)

# Valid & Test: just copy originals, no extra augmented variants
augment_split("valid", n_aug_per_image=0)
augment_split("test",  n_aug_per_image=0)

# ===============================================================
# 7) Write new YAML pointing to augmented dataset
#     - mirrors nc/names from original DATA to keep classes consistent
# ===============================================================
with open(DATA, "r") as f:
    cfg = yaml.safe_load(f)

cfg_aug = dict(cfg)  # shallow copy is fine for this flat schema
cfg_aug["path"]  = str(AUG_SPLIT)     # root of augmented dataset
cfg_aug["train"] = "train/images"
cfg_aug["val"]   = "valid/images"
cfg_aug["test"]  = "test/images"
# keep cfg_aug["nc"] and cfg_aug["names"] as-is from DATA

DATA_AUG = WORK / "data_simclr_augmented.yaml"
with open(DATA_AUG, "w") as f:
    yaml.safe_dump(cfg_aug, f, sort_keys=False)

# quick summary
def split_stats(root: Path, split: str):
    img_dir = root / split / "images"
    lbl_dir = root / split / "labels"
    imgs = sum(1 for _ in img_dir.rglob("*") if _.suffix in IMG_EXTS)
    lbls = sum(1 for _ in lbl_dir.glob("*.txt"))
    return imgs, lbls

print("\n=== Summary ===")
for sp in ["train", "valid", "test"]:
    oi, ol = split_stats(SPLIT, sp)
    ai, al = split_stats(AUG_SPLIT, sp)
    print(f"{sp:>5} | orig: {oi:5d} imgs / {ol:5d} txt  ->  aug: {ai:5d} imgs / {al:5d} txt")

print("\nAugmented yaml written to:", DATA_AUG)
print("✓ Augmentation finished.")


## Before/After augmentation plots

In [ ]:
# ==== Before/After augmentation plots ====
# Reads from your paths:
#   SPLIT     = /kaggle/working/palm_simclr_ssl/0_yolo_split
#   AUG_SPLIT = /kaggle/working/palm_simclr_ssl/0_yolo_split_aug
# Produces: split sizes, class distribution (train), bbox area/AR, per-image box counts.

from pathlib import Path
import yaml
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

BASE      = Path("/kaggle/working")
WORK      = BASE / "palm_simclr_ssl"
SPLIT     = WORK / "0_yolo_split"
AUG_SPLIT = WORK / "0_yolo_split_aug"
DATA      = WORK / "data_simclr.yaml"
IMG_EXTS  = (".jpg", ".jpeg", ".png", ".JPG", ".PNG")

def _warn(msg): print(f"[WARN] {msg}")

if not SPLIT.exists() or not AUG_SPLIT.exists() or not DATA.exists():
    _warn(f"Expected paths missing.\n  SPLIT={SPLIT.exists()}  AUG_SPLIT={AUG_SPLIT.exists()}  DATA={DATA.exists()}")
    _warn("Run your COCO→YOLO conversion and the augmentation script first, then re-run this cell.")
else:
    with open(DATA, "r") as f:
        cfg = yaml.safe_load(f)
    names = cfg.get("names", [])

    def read_yolo_txt(p: Path):
        out = []
        if p.exists():
            for line in p.read_text().splitlines():
                s = line.strip().split()
                if len(s) == 5:
                    try:
                        cls = int(s[0]); xc, yc, w, h = map(float, s[1:])
                        if 0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1:
                            out.append((cls, xc, yc, w, h))
                    except: pass
        return out

    def scan_split(root: Path, split: str):
        img_dir = root / split / "images"
        lbl_dir = root / split / "labels"
        imgs = [p for p in img_dir.rglob("*") if p.suffix in IMG_EXTS]
        per_img = []
        cls_ct  = Counter()
        areas   = []
        aspects = []
        for ip in imgs:
            lp = lbl_dir / (ip.stem + ".txt")
            bxs = read_yolo_txt(lp)
            per_img.append(len(bxs))
            for cls, xc, yc, w, h in bxs:
                cls_ct[cls] += 1
                areas.append(w*h)
                if h > 0: aspects.append(w/h)
        return dict(n_images=len(imgs),
                    n_label_files=sum(1 for _ in (root/split/"labels").glob("*.txt")) if (root/split/"labels").exists() else 0,
                    per_image_counts=per_img,
                    class_counter=cls_ct,
                    areas=areas,
                    aspects=aspects)

    orig_train = scan_split(SPLIT, "train")
    aug_train  = scan_split(AUG_SPLIT, "train")
    orig_val   = scan_split(SPLIT, "valid")
    aug_val    = scan_split(AUG_SPLIT, "valid")
    orig_test  = scan_split(SPLIT, "test")
    aug_test   = scan_split(AUG_SPLIT, "test")

    # 1) Images per split
    splits = ["train", "valid", "test"]
    orig_counts = [orig_train["n_images"], orig_val["n_images"], orig_test["n_images"]]
    aug_counts  = [aug_train["n_images"],  aug_val["n_images"],  aug_test["n_images"]]
    x = np.arange(len(splits)); w = 0.35
    plt.figure(figsize=(9,5))
    plt.bar(x - w/2, orig_counts, width=w, label="original")
    plt.bar(x + w/2, aug_counts,  width=w, label="augmented")
    plt.xticks(x, splits); plt.ylabel("# images"); plt.title("Images per split (before vs after)")
    plt.legend(); plt.tight_layout(); plt.show()

    # 2) Class distribution (train)
    all_cls = sorted(set(orig_train["class_counter"]) | set(aug_train["class_counter"]))
    orig_vals = [orig_train["class_counter"].get(i, 0) for i in all_cls]
    aug_vals  = [aug_train["class_counter"].get(i, 0) for i in all_cls]
    plt.figure(figsize=(max(10, len(all_cls)*0.7), 5))
    x = np.arange(len(all_cls))
    plt.bar(x - w/2, orig_vals, width=w, label="original")
    plt.bar(x + w/2, aug_vals,  width=w, label="augmented")
    tick_labels = [names[i] if i < len(names) else str(i) for i in all_cls]
    plt.xticks(x, tick_labels, rotation=45, ha="right")
    plt.ylabel("# boxes"); plt.title("Class distribution (train) — before vs after")
    plt.legend(); plt.tight_layout(); plt.show()

    # 3) BBox area distribution (train)
    plt.figure(figsize=(9,5))
    plt.hist(orig_train["areas"], bins=40, alpha=0.7, label="original")
    plt.hist(aug_train["areas"],  bins=40, alpha=0.7, label="augmented")
    plt.xlabel("normalized area (w*h)"); plt.ylabel("frequency")
    plt.title("BBox area distribution (train)"); plt.legend(); plt.tight_layout(); plt.show()

    # 4) Aspect ratio distribution (train)
    plt.figure(figsize=(9,5))
    orig_ar = [a for a in orig_train["aspects"] if np.isfinite(a) and a < 10]
    aug_ar  = [a for a in aug_train["aspects"]  if np.isfinite(a) and a < 10]
    plt.hist(orig_ar, bins=40, alpha=0.7, label="original")
    plt.hist(aug_ar,  bins=40, alpha=0.7, label="augmented")
    plt.xlabel("aspect ratio (w/h)"); plt.ylabel("frequency")
    plt.title("BBox aspect ratio distribution (train) — truncated at 10x")
    plt.legend(); plt.tight_layout(); plt.show()

    # 5) Per-image bbox counts (train)
    plt.figure(figsize=(9,5))
    max_bins = max(orig_train["per_image_counts"] + aug_train["per_image_counts"] + [1])
    bins = np.arange(0, max_bins + 2) - 0.5
    plt.hist(orig_train["per_image_counts"], bins=bins, alpha=0.7, label="original")
    plt.hist(aug_train["per_image_counts"],  bins=bins, alpha=0.7, label="augmented")
    plt.xlabel("# boxes per image"); plt.ylabel("# images")
    plt.title("Per-image bbox counts (train)"); plt.legend(); plt.tight_layout(); plt.show()

    # Quick numeric summary
    def row(tag, d):
        return dict(split=tag,
                    images=d["n_images"],
                    label_files=d["n_label_files"],
                    total_boxes=sum(d["per_image_counts"]),
                    boxes_per_img_mean=round((sum(d["per_image_counts"])/d["n_images"]) if d["n_images"] else 0.0, 4))
    summary = [
        row("orig-train", orig_train), row("aug-train",  aug_train),
        row("orig-valid", orig_val),   row("aug-valid",  aug_val),
        row("orig-test",  orig_test),  row("aug-test",   aug_test),
    ]
    print("\n=== Dataset Summary (Before vs After) ===")
    for r in summary: print(r)


## Side-by-side viewer: original vs augmented with bboxes

In [ ]:
# ==== Side-by-side viewer: original vs augmented with bboxes ====
from pathlib import Path
import random, cv2
import matplotlib.pyplot as plt

BASE      = Path("/kaggle/working")
WORK      = BASE / "palm_simclr_ssl"
SPLIT     = WORK / "0_yolo_split"
AUG_SPLIT = WORK / "0_yolo_split_aug"
IMG_EXTS  = (".jpg", ".jpeg", ".png", ".JPG", ".PNG")

def read_yolo(lbl_path):
    boxes = []
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            p = line.strip().split()
            if len(p)==5:
                cls = int(p[0]); xc, yc, w, h = map(float, p[1:])
                boxes.append((cls, xc, yc, w, h))
    return boxes

def draw_boxes(img_bgr, boxes, color=(5,180,255), thickness=2):
    h, w = img_bgr.shape[:2]
    img = img_bgr.copy()
    for cls, xc, yc, bw, bh in boxes:
        x1 = int((xc - bw/2) * w); y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w); y2 = int((yc + bh/2) * h)
        cv2.rectangle(img, (x1,y1), (x2,y2), color, thickness)
        cv2.putText(img, str(cls), (x1, max(0,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    return img

# pick a random train image that has at least one augmented sibling
orig_imgs = [p for p in (SPLIT/"train"/"images").rglob("*") if p.suffix in IMG_EXTS]
candidates = []
for p in orig_imgs:
    stem = p.stem
    aug_label_dir = AUG_SPLIT/"train"/"labels"
    # look for any file like <stem>_augX.txt in augmented labels
    aug_sibs = list(aug_label_dir.glob(f"{stem}_aug*.txt"))
    if len(aug_sibs) > 0:
        candidates.append((p, aug_sibs))

if not candidates:
    print("No augmented siblings found. Make sure augmentation has run.")
else:
    orig_img_path, aug_lbls = random.choice(candidates)
    # pick one augmented sample
    aug_lbl_path = random.choice(aug_lbls)
    aug_img_path = (AUG_SPLIT/"train"/"images"/(aug_lbl_path.stem + ".jpg"))
    if not aug_img_path.exists():
        # try png fallback
        aug_img_path = (AUG_SPLIT/"train"/"images"/(aug_lbl_path.stem + ".png"))

    # load images + labels
    orig_lbl_path = SPLIT/"train"/"labels"/(orig_img_path.stem + ".txt")
    orig_boxes = read_yolo(orig_lbl_path)
    aug_boxes  = read_yolo(aug_lbl_path)

    orig_bgr = cv2.imread(str(orig_img_path))
    aug_bgr  = cv2.imread(str(aug_img_path))
    if orig_bgr is None or aug_bgr is None:
        print("Could not read images; check file paths.")
    else:
        orig_vis = cv2.cvtColor(draw_boxes(orig_bgr, orig_boxes), cv2.COLOR_BGR2RGB)
        aug_vis  = cv2.cvtColor(draw_boxes(aug_bgr,  aug_boxes),  cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(12,6))
        plt.subplot(1,2,1); plt.imshow(orig_vis); plt.axis("off"); plt.title(f"Original: {orig_img_path.name}")
        plt.subplot(1,2,2); plt.imshow(aug_vis);  plt.axis("off"); plt.title(f"Augmented: {aug_img_path.name}")
        plt.tight_layout(); plt.show()


## SimCLR dataset & augmentations 

In [ ]:
# ===============================================================
# SimCLR Dataset & Augmentations
# ===============================================================
from pathlib import Path
from PIL import Image, ImageFilter
import yaml
import random

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

BASE        = Path("/kaggle/working")
WORK        = BASE / "palm_simclr_ssl"
DATA_AUG    = WORK / "data_simclr_augmented.yaml"   # from your previous code
SIMCLR_SSL_W = WORK / "simclr_backbone_with_hooks_yolov12s.pt"

SSL_IMG_SIZE = 320   # multiple of 32 so YOLO is happy
IMG_EXTS = (".jpg", ".jpeg", ".png", ".JPG", ".PNG")

# --------- SimCLR-style augmentations ---------
class GaussianBlur(object):
    """Implements Gaussian blur as used in SimCLR."""
    def __init__(self, kernel_size, sigma=(0.1, 2.0)):
        self.kernel_size = kernel_size
        self.sigma = sigma

    def __call__(self, x):
        sigma = random.uniform(self.sigma[0], self.sigma[1])
        return x.filter(ImageFilter.GaussianBlur(radius=sigma))


class SimCLRTransform:
    """
    Returns two strongly augmented views of the same image.
    We DON'T do mean/std normalization so inputs stay in 0..1 like YOLO.
    """
    def __init__(self, size=SSL_IMG_SIZE):
        color_jitter = transforms.ColorJitter(
            brightness=0.8,
            contrast=0.8,
            saturation=0.8,
            hue=0.2,
        )

        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size=size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([color_jitter], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            GaussianBlur(kernel_size=int(0.1 * size)),
            transforms.ToTensor(),  # -> [0,1] tensor
        ])

    def __call__(self, x):
        xi = self.transform(x)
        xj = self.transform(x)
        return xi, xj


class SimCLRDataset(Dataset):
    """
    Reads images from the YOLO data yaml and returns (view1, view2).
    Uses the *augmented* dataset you created (data_simclr_augmented.yaml).
    """
    def __init__(self, data_yaml, split="train", transform=None):
        self.data_yaml = Path(data_yaml)
        with open(self.data_yaml, "r") as f:
            cfg = yaml.safe_load(f)

        self.root = Path(cfg["path"])
        # e.g. cfg["train"] = "train/images"
        rel = cfg[split]
        self.img_dir = self.root / rel
        self.transform = transform or SimCLRTransform(size=SSL_IMG_SIZE)

        self.img_paths = [
            p for p in self.img_dir.rglob("*")
            if p.suffix in IMG_EXTS
        ]
        self.img_paths.sort()
        print(f"SimCLRDataset[{split}] -> {len(self.img_paths)} images from {self.img_dir}")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        path = self.img_paths[idx]
        img = Image.open(path).convert("RGB")
        x1, x2 = self.transform(img)
        return x1, x2

# ---- create DataLoader for SimCLR on the augmented train split ----
simclr_transform = SimCLRTransform(size=SSL_IMG_SIZE)
simclr_train_ds = SimCLRDataset(DATA_AUG, split="train", transform=simclr_transform)

BATCH_SIZE = 64   # adjust if you get OOM
simclr_train_loader = DataLoader(
    simclr_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,   # important for SimCLR (2N pairs)
)
print("SimCLR train batches:", len(simclr_train_loader))


## YOLOv12s backbone wrapper + SimCLR projection head

In [ ]:
# ===============================================================
# YOLOv12s backbone + SimCLR projection head
# ===============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO

from pathlib import Path

MODEL_CFG   = "yolo12s.pt"           # you already defined this earlier
YOL0V12S_W  = WORK / "yolov12s.pt"      # optional pretrained weights

device = "cuda" if torch.cuda.is_available() else "cpu"
print("SimCLR device:", device)

# ----- Try to import Detect head for robust backbone index discovery -----
try:
    from ultralytics.nn.modules.head import Detect
except ImportError:
    try:
        from ultralytics.nn.modules import Detect  # older / different versions
    except ImportError:
        Detect = None


def find_backbone_idx(det_model):
    """
    det_model: underlying ultralytics DetectionModel (yolo.model)
    Returns index of the last backbone block (the one before Detect).
    """
    if hasattr(det_model, "model"):
        modules = det_model.model  # ModuleList
    else:
        modules = list(det_model.children())

    if Detect is not None:
        for i, m in enumerate(modules):
            if isinstance(m, Detect):
                if i == 0:
                    raise ValueError("Detect layer is first? Unexpected, can't pick backbone index.")
                print(f"Detect head found at idx {i}, using backbone idx {i-1}")
                return i - 1

    # Fallback: use second-to-last module
    print("Detect not found; falling back to module index -2 as backbone.")
    return len(modules) - 2


class YoloSimCLREncoder(nn.Module):
    """
    Wraps the YOLO detection model, attaches a forward hook to a backbone layer,
    and adds a SimCLR projection head.
    """
    def __init__(self, det_model, img_size=SSL_IMG_SIZE,
                 proj_dim=128, hidden_dim=2048, device="cuda"):
        super().__init__()
        self.det_model = det_model
        self.img_size = img_size
        self._feat = None

        # 1) Register hook on backbone output
        backbone_idx = find_backbone_idx(self.det_model)
        target_module = self.det_model.model[backbone_idx]
        print(f"Registering SimCLR hook on layer idx {backbone_idx}: {target_module.__class__.__name__}")

        def hook_fn(module, inp, out):
            # out: [B, C, H, W]
            self._feat = out

        target_module.register_forward_hook(hook_fn)

        # 2) Run a dummy forward to infer feature dimension
        with torch.no_grad():
            dummy = torch.zeros(1, 3, img_size, img_size, device=device)
            _ = self.det_model(dummy)
            feat = self._feat
        if feat is None:
            raise RuntimeError("Hook did not capture any features. Check backbone_idx or YOLO version.")

        feat_dim = feat.shape[1]
        print("Backbone feature dim (channels):", feat_dim)

        # 3) Projection head g(.)
        self.projection_head = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, proj_dim),
        )

    def forward(self, x, return_backbone=False):
        """
        x: [B, 3, H, W] in [0,1]
        returns: projection z (and optionally pooled backbone features h)
        """
        _ = self.det_model(x)          # triggers hook
        feat = self._feat              # [B, C, H, W]
        pooled = feat.mean(dim=(2, 3)) # global average pool -> [B, C]
        z = self.projection_head(pooled)  # [B, proj_dim]
        if return_backbone:
            return pooled, z
        return z


# ----- Instantiate YOLO and wrap it for SimCLR -----
# Try to load from weights if exists, otherwise from YAML
if YOL0V12S_W.exists():
    print("Loading YOLO from weights:", YOL0V12S_W)
    yolo = YOLO(str(YOL0V12S_W))
else:
    print("Loading YOLO from config:", MODEL_CFG)
    yolo = YOLO(MODEL_CFG)

det_model = yolo.model.to(device)  # underlying nn.Module

# Wrap with SimCLR encoder
simclr_encoder = YoloSimCLREncoder(
    det_model=det_model,
    img_size=SSL_IMG_SIZE,
    proj_dim=128,       # typical SimCLR projection dim
    hidden_dim=2048,
    device=device,
).to(device)

print("YoloSimCLREncoder parameters:", sum(p.numel() for p in simclr_encoder.parameters()) / 1e6, "M")


## SimCLR loss + training loop + saving simclr_backbone_with_hooks_yolov12s.pt

In [ ]:
# ===============================================================
# 0) Imports & paths
# ===============================================================
from pathlib import Path
from PIL import Image, ImageFilter
import yaml
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from ultralytics import YOLO
from contextlib import nullcontext

# ---- base paths (consistent with your previous code) ----
BASE        = Path("/kaggle/working")
WORK        = BASE / "palm_simclr_ssl"

DATA_AUG         = WORK / "data_simclr_augmented.yaml"   # preferred
DATA_FALLBACK    = WORK / "data_simclr.yaml"             # fallback
SIMCLR_SSL_W     = WORK / "simclr_backbone_with_hooks_yolov12s.pt"
YOL0V12S_W       = WORK / "yolov12s.pt"
MODEL_CFG        = "yolo12s.pt"   # ✅ correct for your setup

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ===============================================================
# 1) Choose data yaml (augmented → fallback to original)
# ===============================================================
if not DATA_AUG.exists():
    print("⚠️ data_simclr_augmented.yaml not found at:", DATA_AUG)
    if DATA_FALLBACK.exists():
        print("   Falling back to original yaml:", DATA_FALLBACK)
        DATA_YAML = DATA_FALLBACK
    else:
        raise FileNotFoundError(
            f"Neither augmented nor original data yaml found:\n"
            f"  {DATA_AUG}\n  {DATA_FALLBACK}\n"
            f"Re-run your COCO→YOLO + augmentation code first."
        )
else:
    DATA_YAML = DATA_AUG
    print("Using augmented data yaml:", DATA_YAML)

# ===============================================================
# 2) SimCLR Dataset & Augmentations
# ===============================================================
SSL_IMG_SIZE = 320   # multiple of 32 so YOLO is happy
IMG_EXTS = (".jpg", ".jpeg", ".png", ".JPG", ".PNG")

class GaussianBlur(object):
    """Implements Gaussian blur as used in SimCLR."""
    def __init__(self, kernel_size, sigma=(0.1, 2.0)):
        self.kernel_size = kernel_size
        self.sigma = sigma

    def __call__(self, x):
        sigma = random.uniform(self.sigma[0], self.sigma[1])
        return x.filter(ImageFilter.GaussianBlur(radius=sigma))


class SimCLRTransform:
    """
    Returns two strongly augmented views of the same image.
    We DON'T do mean/std normalization so inputs stay in 0..1 like YOLO.
    """
    def __init__(self, size=SSL_IMG_SIZE):
        color_jitter = transforms.ColorJitter(
            brightness=0.8,
            contrast=0.8,
            saturation=0.8,
            hue=0.2,
        )

        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size=size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([color_jitter], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            GaussianBlur(kernel_size=int(0.1 * size)),
            transforms.ToTensor(),  # -> [0,1] tensor
        ])

    def __call__(self, x):
        xi = self.transform(x)
        xj = self.transform(x)
        return xi, xj


class SimCLRDataset(Dataset):
    """
    Reads images from the YOLO data yaml and returns (view1, view2).
    Uses the augmented dataset if available, else original.
    """
    def __init__(self, data_yaml, split="train", transform=None):
        self.data_yaml = Path(data_yaml)
        with open(self.data_yaml, "r") as f:
            cfg = yaml.safe_load(f)

        self.root = Path(cfg["path"])
        rel = cfg[split]        # e.g. "train/images"
        self.img_dir = self.root / rel
        self.transform = transform or SimCLRTransform(size=SSL_IMG_SIZE)

        self.img_paths = [
            p for p in self.img_dir.rglob("*")
            if p.suffix in IMG_EXTS
        ]
        self.img_paths.sort()
        print(f"SimCLRDataset[{split}] -> {len(self.img_paths)} images from {self.img_dir}")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        path = self.img_paths[idx]
        img = Image.open(path).convert("RGB")
        x1, x2 = self.transform(img)
        return x1, x2


simclr_transform = SimCLRTransform(size=SSL_IMG_SIZE)
simclr_train_ds = SimCLRDataset(DATA_YAML, split="train", transform=simclr_transform)

BATCH_SIZE = 64   # adjust if you get OOM
simclr_train_loader = DataLoader(
    simclr_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,   # important for SimCLR (2N pairs)
)
print("SimCLR train batches:", len(simclr_train_loader))

# ===============================================================
# 3) YOLOv12s backbone + SimCLR projection head
# ===============================================================
# Try to import Detect head for robust backbone index discovery
try:
    from ultralytics.nn.modules.head import Detect
except Exception:
    try:
        from ultralytics.nn.modules import Detect
    except Exception:
        Detect = None

def find_backbone_idx(det_model):
    """
    det_model: underlying ultralytics DetectionModel (yolo.model)
    Returns index of the last backbone block (the one before Detect).
    """
    if hasattr(det_model, "model"):
        modules = det_model.model  # ModuleList
    else:
        modules = list(det_model.children())

    if Detect is not None:
        for i, m in enumerate(modules):
            if isinstance(m, Detect):
                if i == 0:
                    raise ValueError("Detect layer is first? Unexpected, can't pick backbone index.")
                print(f"Detect head found at idx {i}, using backbone idx {i-1}")
                return i - 1

    # Fallback: use second-to-last module
    print("Detect not found; falling back to module index -2 as backbone.")
    return len(modules) - 2


class YoloSimCLREncoder(nn.Module):
    """
    Wraps the YOLO detection model, attaches a forward hook to a backbone layer,
    and adds a SimCLR projection head.
    """
    def __init__(self, det_model, img_size=SSL_IMG_SIZE,
                 proj_dim=128, hidden_dim=2048, device="cuda"):
        super().__init__()
        self.det_model = det_model
        self.img_size = img_size
        self._feat = None

        # 1) Register hook on backbone output
        backbone_idx = find_backbone_idx(self.det_model)
        target_module = self.det_model.model[backbone_idx]
        print(f"Registering SimCLR hook on layer idx {backbone_idx}: {target_module.__class__.__name__}")

        def hook_fn(module, inp, out):
            # out: [B, C, H, W]
            self._feat = out

        target_module.register_forward_hook(hook_fn)

        # 2) Run a dummy forward to infer feature dimension
        with torch.no_grad():
            dummy = torch.zeros(1, 3, img_size, img_size, device=device)
            _ = self.det_model(dummy)
            feat = self._feat
        if feat is None:
            raise RuntimeError("Hook did not capture any features. Check backbone_idx or YOLO version.")

        feat_dim = feat.shape[1]
        print("Backbone feature dim (channels):", feat_dim)

        # 3) Projection head g(.)
        self.projection_head = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, proj_dim),
        )

    def forward(self, x, return_backbone=False):
        """
        x: [B, 3, H, W] in [0,1]
        returns: projection z (and optionally pooled backbone features h)
        """
        _ = self.det_model(x)          # triggers hook
        feat = self._feat              # [B, C, H, W]
        pooled = feat.mean(dim=(2, 3)) # global average pool -> [B, C]
        z = self.projection_head(pooled)  # [B, proj_dim]
        if return_backbone:
            return pooled, z
        return z


# Instantiate YOLO and wrap it for SimCLR
if YOL0V12S_W.exists():
    print("Loading YOLO from weights:", YOL0V12S_W)
    yolo = YOLO(str(YOL0V12S_W))
else:
    print("Loading YOLO from config/weights:", MODEL_CFG)
    yolo = YOLO(MODEL_CFG)

det_model = yolo.model.to(device)

simclr_encoder = YoloSimCLREncoder(
    det_model=det_model,
    img_size=SSL_IMG_SIZE,
    proj_dim=128,
    hidden_dim=2048,
    device=device,
).to(device)

print("YoloSimCLREncoder parameters:",
      sum(p.numel() for p in simclr_encoder.parameters()) / 1e6, "M")

# ===============================================================
# 4) SimCLR NT-Xent loss (with overflow-safe masking)
# ===============================================================
def nt_xent_loss(z_i, z_j, temperature=0.2):
    """
    Standard NT-Xent loss from SimCLR.
    z_i, z_j: [N, d] projections for the two views.
    """
    batch_size = z_i.size(0)
    z = torch.cat([z_i, z_j], dim=0)           # [2N, d]
    z = F.normalize(z, dim=1)                  # L2-normalize

    # Cosine similarity matrix [2N, 2N]
    sim = torch.matmul(z, z.T) / temperature

    # Mask self-similarity using minimum representable value for dtype
    mask = torch.eye(2 * batch_size, device=z.device, dtype=torch.bool)
    min_val = torch.finfo(sim.dtype).min       # safe for float16/32
    sim = sim.masked_fill(mask, min_val)

    # Positive index for each sample: (i + N) % (2N)
    labels = torch.arange(2 * batch_size, device=z.device)
    labels = (labels + batch_size) % (2 * batch_size)

    loss = F.cross_entropy(sim, labels)
    return loss

# ===============================================================
# 5) AMP setup (torch.amp API with fallback, NO device_type kwarg)
# ===============================================================
use_amp = (device == "cuda")

if use_amp and hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
    # Your torch.amp.GradScaler does not accept device_type kwarg
    scaler = torch.amp.GradScaler(enabled=True)

    def autocast():
        # pass "cuda" positionally
        return torch.amp.autocast("cuda", enabled=True)

elif use_amp:
    # Fallback to old cuda.amp API if torch.amp is not available
    from torch.cuda.amp import GradScaler as OldGradScaler, autocast as old_autocast
    scaler = OldGradScaler(enabled=True)

    def autocast():
        return old_autocast(enabled=True)

else:
    scaler = None

    def autocast():
        return nullcontext()


# 6) Train SimCLR on YOLOv12s backbone
EPOCHS       = 70        
TEMPERATURE  = 0.2
LR           = 1e-3
WEIGHT_DECAY = 1e-4



optimizer = torch.optim.AdamW(
    simclr_encoder.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

simclr_encoder.train()

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    num_steps = 0

    for step, (x1, x2) in enumerate(simclr_train_loader):
        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            z1 = simclr_encoder(x1)  # [B, proj_dim]
            z2 = simclr_encoder(x2)  # [B, proj_dim]
            loss = nt_xent_loss(z1, z2, temperature=TEMPERATURE)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        epoch_loss += loss.item()
        num_steps  += 1

        if (step + 1) % 50 == 0:
            print(f"Epoch [{epoch}/{EPOCHS}] Step [{step+1}/{len(simclr_train_loader)}] "
                  f"Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / max(1, num_steps)
    print(f"==== Epoch [{epoch}/{EPOCHS}] Avg SimCLR loss: {avg_loss:.4f} ====")

    # Optional: intermediate checkpoints
    if epoch % 10 == 0:
        ckpt_path = WORK / f"simclr_yolov12s_epoch{epoch}.pt"
        torch.save(
            {
                "epoch": epoch,
                "det_model_state_dict": simclr_encoder.det_model.state_dict(),
                "projection_head_state_dict": simclr_encoder.projection_head.state_dict(),
            },
            ckpt_path,
        )
        print("Saved intermediate checkpoint to:", ckpt_path)

# ===============================================================
# 7) Save final backbone weights for later detection fine-tuning
# ===============================================================
torch.save(
    {
        "det_model_state_dict": simclr_encoder.det_model.state_dict(),
        "projection_head_state_dict": simclr_encoder.projection_head.state_dict(),
    },
    SIMCLR_SSL_W,
)
print("✓ Final SimCLR backbone weights saved to:", SIMCLR_SSL_W)


## Fine-tune for detection

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

# ---------------- Paths (same as before) ----------------
BASE        = Path("/kaggle/working")
WORK        = BASE / "palm_simclr_ssl"

DATA_AUG      = WORK / "data_simclr_augmented.yaml"   # augmented YOLO data
DATA_FALLBACK = WORK / "data_simclr.yaml"             # original, if needed

SIMCLR_SSL_W  = WORK / "simclr_backbone_with_hooks_yolov12s.pt"
MODEL_CFG     = "yolo12s.pt"                          # your YOLO backbone+head

# ✅ keep device as string like before
device = "cuda" if torch.cuda.is_available() else "cpu"
yolo_device = 0 if device == "cuda" else "cpu"   # what we pass into YOLO

print("Device:", device)

# ------------ pick which data yaml to use ------------
if DATA_AUG.exists():
    DATA_YAML = DATA_AUG          # ✅ same name style as SimCLR part
    print("Using augmented data yaml for detection:", DATA_YAML)
elif DATA_FALLBACK.exists():
    DATA_YAML = DATA_FALLBACK
    print("⚠️ Augmented yaml not found, using original:", DATA_YAML)
else:
    raise FileNotFoundError("No data yaml found for detection training.")

# (optional) also expose as DATA for later val() code
DATA = DATA_YAML

# ------------ load YOLO detection model ------------
model = YOLO(MODEL_CFG)   # loads yolo12s.pt (same arch as SimCLR training)

# ------------ load SimCLR-pretrained weights ------------
ckpt = torch.load(SIMCLR_SSL_W, map_location="cpu")
simclr_state = ckpt["det_model_state_dict"]

missing, unexpected = model.model.load_state_dict(simclr_state, strict=False)
print("Loaded SimCLR weights into YOLO model.")
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))
print("Example missing keys:", missing[:5])
print("Example unexpected keys:", unexpected[:5])


# Fine-tune for detection 

results = model.train(
    data=str(DATA_YAML),   
    epochs=70,            
    imgsz=640,
    batch=16,
    device=yolo_device,    
    project=str(WORK),
    name="yolo12s_simclr_det",
    pretrained=False,      
    patience=7,           
    lr0=0.005,             
    weight_decay=6e-4,     
)

print("Training finished. Best weights saved at:")
print(WORK / "yolo12s_simclr_det" / "weights" / "best.pt")

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- Paths ----------------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"
run_dir = WORK / "yolo12s_simclr_det"   
results_csv = run_dir / "results.csv"

print("Reading:", results_csv)
df = pd.read_csv(results_csv)
print("Columns in results.csv:", df.columns.tolist())

# x-axis (epoch)
if "epoch" in df.columns:
    epochs = df["epoch"]
else:
    epochs = df.index

# ==============================
# 1) Train + Val box loss
# ==============================
plt.figure(figsize=(8, 5), dpi=150)

plt.plot(epochs, df["train/box_loss"], label="Train box loss", linewidth=2)

if "val/box_loss" in df.columns:
    plt.plot(epochs, df["val/box_loss"], label="Val box loss", linewidth=2)

plt.xlabel("Epoch")
plt.ylabel("Box loss")
plt.title("Training and Validation Box Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
from ultralytics import YOLO
from pathlib import Path
import torch

# ========= 0) Paths & model loading =========
BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"

det_run_name = "yolo12s_simclr_det"  # name used in model.train(...)
run_dir = WORK / det_run_name
best_pt = run_dir / "weights" / "best.pt"

print("\nEvaluating …")
print("Best checkpoint:", best_pt)

best_model = YOLO(str(best_pt))

# Use GPU index 0 if available, otherwise CPU/None
eval_device = 0 if torch.cuda.is_available() else None

# If you already have `data_yaml` from training, reuse it.
# Otherwise, reconstruct it like this:
try:
    DATA = data_yaml
except NameError:
    DATA_AUG = WORK / "data_simclr_augmented.yaml"
    DATA_FALLBACK = WORK / "data_simclr.yaml"
    if DATA_AUG.exists():
        DATA = DATA_AUG
        print("Using augmented data yaml:", DATA)
    elif DATA_FALLBACK.exists():
        DATA = DATA_FALLBACK
        print("Using fallback data yaml:", DATA)
    else:
        raise FileNotFoundError("No data yaml found (augmented or original).")


# ========= 1) Confusion-matrix helpers =========
def confusion_stats_from_matrix(cm: np.ndarray):
    """
    cm: confusion matrix (Ultralytics style)
        Shape: (nc+1, nc+1) -> last row/col = background
    Returns:
        sens_macro, spec_macro, kappa
    """
    # drop background row/col if present
    if cm.shape[0] > 1 and cm.shape[0] == cm.shape[1]:
        cm = cm[:-1, :-1]

    total = cm.sum()
    if total == 0:
        return float("nan"), float("nan"), float("nan")

    diag = np.diag(cm)
    row_sum = cm.sum(axis=1)
    col_sum = cm.sum(axis=0)

    # Observed agreement
    p0 = diag.sum() / total

    # Expected agreement
    pe = (row_sum * col_sum).sum() / (total ** 2)

    # Cohen's kappa
    if 1 - pe == 0:
        kappa = float("nan")
    else:
        kappa = (p0 - pe) / (1 - pe)

    # Per-class sensitivity/specificity
    sens_list = []
    spec_list = []
    for i in range(cm.shape[0]):
        TP = cm[i, i]
        FN = row_sum[i] - TP
        FP = col_sum[i] - TP
        TN = total - (TP + FN + FP)

        sens = TP / (TP + FN) if (TP + FN) > 0 else float("nan")
        spec = TN / (TN + FP) if (TN + FP) > 0 else float("nan")

        sens_list.append(sens)
        spec_list.append(spec)

    sens_macro = float(np.nanmean(sens_list))
    spec_macro = float(np.nanmean(spec_list))

    return sens_macro, spec_macro, kappa


def print_metrics(name, metrics):
    print(f"\n{name} metrics")

    # ---------- main detection metrics + PR-AUC ----------
    try:
        # Newer Ultralytics API
        mp      = metrics.box.mp
        mr      = metrics.box.mr
        map50   = metrics.box.map50
        map5095 = metrics.box.map

        # PR-AUC = AP = area under P–R curve
        pr_auc_50   = map50     # mean PR-AUC @ IoU 0.5
        pr_auc_5095 = map5095   # mean PR-AUC @ IoU 0.5:0.95

        mf1 = (2 * mp * mr) / (mp + mr) if (mp + mr) > 0 else 0.0

        print(f"Precision (mP)             : {mp:.4f}")
        print(f"Recall (mR, Sensitivity)   : {mr:.4f}")
        print(f"F1-Score                   : {mf1:.4f}")
        print(f"mAP@0.50 (PR-AUC@0.5)      : {map50:.4f}")
        print(f"mAP@0.50:0.95 (mean PR-AUC): {map5095:.4f}")

    except AttributeError:
        # Older API fallback
        try:
            mp, mr, map50, map5095 = metrics.mean_results()
            mf1 = (2 * mp * mr) / (mp + mr) if (mp + mr) > 0 else 0.0

            pr_auc_50   = map50
            pr_auc_5095 = map5095

            print(f"Precision (mP)             : {mp:.4f}")
            print(f"Recall (mR, Sensitivity)   : {mr:.4f}")
            print(f"F1-Score                   : {mf1:.4f}")
            print(f"mAP@0.50 (PR-AUC@0.5)      : {map50:.4f}")
            print(f"mAP@0.50:0.95 (mean PR-AUC): {map5095:.4f}")
        except Exception:
            print("Raw metrics object (API changed):")
            print(metrics)
            pr_auc_50 = pr_auc_5095 = float("nan")

    # ---------- confusion-matrix based metrics ----------
    try:
        cm = metrics.confusion_matrix.matrix
        cm = np.array(cm, dtype=float)

        sens_macro, spec_macro, kappa = confusion_stats_from_matrix(cm)

        print(f"Sensitivity (macro TPR)    : {sens_macro:.4f}")
        print(f"Specificity (macro TNR)    : {spec_macro:.4f}")
        print(f"Cohen's Kappa              : {kappa:.4f}")
    except Exception as e:
        print("Extra confusion-based metrics unavailable:", e)


# ========= 2) Run VAL and TEST evaluations =========
val_metrics = best_model.val(
    data=str(DATA),
    imgsz=640,
    split="val",
    batch=8,
    device=eval_device,
    verbose=False,
)

test_metrics = best_model.val(
    data=str(DATA),
    imgsz=640,
    split="test",
    batch=8,
    device=eval_device,
    verbose=False,
)

print_metrics("Validation", val_metrics)
print_metrics("Test", test_metrics)


## Random detection grid on test images

In [ ]:
# Random detection grid on test images 
import math
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch

from ultralytics import YOLO

BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"

# 1) Locate test images (prefer original split, fallback to augmented)
SPLIT = WORK / "0_yolo_split"          # original YOLO split
AUG_SPLIT = WORK / "0_yolo_split_aug"  # augmented YOLO split

if SPLIT.exists():
    print("Using original split for visualization:", SPLIT)
else:
    print("⚠️ Original split not found, using augmented split:", AUG_SPLIT)
    SPLIT = AUG_SPLIT

test_img_dir = SPLIT / "test" / "images"
if not test_img_dir.exists():
    raise FileNotFoundError(f"Test image dir not found: {test_img_dir}")

all_test_imgs = sorted(
    list(test_img_dir.glob("*.jpg")) +
    list(test_img_dir.glob("*.png")) +
    list(test_img_dir.glob("*.jpeg"))
)
print(f"Found {len(all_test_imgs)} test images in {test_img_dir}")

n_show = min(2, len(all_test_imgs))   #  show up to 10 images
sample_imgs = random.sample(all_test_imgs, n_show)

# 2) Load best model checkpoint if best_model is not in scope
try:
    best_model
except NameError:
    det_run_name = "yolo12s_simclr_det"
    best_pt = WORK / det_run_name / "weights" / "best.pt"
    print("Loading best model from:", best_pt)
    best_model = YOLO(str(best_pt))

# 3) Run prediction
dev = 0 if torch.cuda.is_available() else None

preds = best_model.predict(
    source=[str(p) for p in sample_imgs],
    imgsz=640,
    conf=0.25,
    device=dev,
    verbose=False,
)

# 4) Show in grid (no filenames in titles)
cols = 5                          # 5x2 grid for 10 images
rows = math.ceil(n_show / cols)
plt.figure(figsize=(4 * cols, 4 * rows))

for i, r in enumerate(preds):
    img_vis = r.plot()  # returns BGR numpy array
    img_vis = cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img_vis)
    # No title (no filename)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ===============================================================
# Random detection grid on test images (10 images, no filenames)
# ===============================================================
import math
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch

from ultralytics import YOLO

BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"

# 1) Locate test images (prefer original split, fallback to augmented)
SPLIT = WORK / "0_yolo_split"          # original YOLO split
AUG_SPLIT = WORK / "0_yolo_split_aug"  # augmented YOLO split

if SPLIT.exists():
    print("Using original split for visualization:", SPLIT)
else:
    print("⚠️ Original split not found, using augmented split:", AUG_SPLIT)
    SPLIT = AUG_SPLIT

test_img_dir = SPLIT / "test" / "images"
if not test_img_dir.exists():
    raise FileNotFoundError(f"Test image dir not found: {test_img_dir}")

all_test_imgs = sorted(
    list(test_img_dir.glob("*.jpg")) +
    list(test_img_dir.glob("*.png")) +
    list(test_img_dir.glob("*.jpeg"))
)
print(f"Found {len(all_test_imgs)} test images in {test_img_dir}")

n_show = min(10, len(all_test_imgs))   
sample_imgs = random.sample(all_test_imgs, n_show)

# 2) Load best model checkpoint if best_model is not in scope
try:
    best_model
except NameError:
    det_run_name = "yolo12s_simclr_det"
    best_pt = WORK / det_run_name / "weights" / "best.pt"
    print("Loading best model from:", best_pt)
    best_model = YOLO(str(best_pt))

# 3) Run prediction
dev = 0 if torch.cuda.is_available() else None

preds = best_model.predict(
    source=[str(p) for p in sample_imgs],
    imgsz=640,
    conf=0.25,
    device=dev,
    verbose=False,
)

# 4) Show in grid (no filenames in titles)
cols = 5                          # 5x2 grid for 10 images
rows = math.ceil(n_show / cols)
plt.figure(figsize=(4 * cols, 4 * rows))

for i, r in enumerate(preds):
    img_vis = r.plot()  # returns BGR numpy array
    img_vis = cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img_vis)
    # No title (no filename)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
import torch
from ultralytics import YOLO

# ---------------------------------------
# 0) Paths and model (reuse existing)
# ---------------------------------------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"

det_run_name = "yolo12s_simclr_det"
best_pt = WORK / det_run_name / "weights" / "best.pt"
print("Best checkpoint:", best_pt)

# If best_model is not defined, load it
try:
    best_model
except NameError:
    best_model = YOLO(str(best_pt))

# DATA yaml (reuse if defined, else detect)
try:
    DATA
except NameError:
    DATA_AUG = WORK / "data_simclr_augmented.yaml"
    DATA_FALLBACK = WORK / "data_simclr.yaml"
    if DATA_AUG.exists():
        DATA = DATA_AUG
        print("Using augmented data yaml:", DATA)
    elif DATA_FALLBACK.exists():
        DATA = DATA_FALLBACK
        print("Using fallback data yaml:", DATA)
    else:
        raise FileNotFoundError("No data yaml found (augmented or original).")

# Load class names from yaml
with open(DATA, "r") as f:
    cfg = yaml.safe_load(f)
class_names = cfg.get("names", [])
nc = len(class_names)

# Optionally add a background label for plotting if needed
class_names_with_bg = class_names + ["background"]

# ---------------------------------------
# 1) Get val & test metrics (if needed)
# ---------------------------------------
eval_device = 0 if torch.cuda.is_available() else None

try:
    val_metrics
    test_metrics
except NameError:
    print("Running val() to obtain metrics (val & test)...")
    val_metrics = best_model.val(
        data=str(DATA),
        imgsz=640,
        split="val",
        batch=8,
        device=eval_device,
        verbose=False,
    )

    test_metrics = best_model.val(
        data=str(DATA),
        imgsz=640,
        split="test",
        batch=8,
        device=eval_device,
        verbose=False,
    )

# ---------------------------------------
# 2) Helper for plotting confusion matrix
# ---------------------------------------
def plot_confusion_matrix(metrics, class_names, title="Confusion Matrix", normalize=True):
    """
    metrics: Ultralytics metrics object from model.val()
    class_names: list of class names (without background); background will be handled if present
    """
    cm = metrics.confusion_matrix.matrix  # (nc+1, nc+1) with bg
    cm = np.array(cm, dtype=float)

    # keep background row/col if you want, or drop them
    # here we'll keep them and use class_names_with_bg for labels
    labels = class_names + ["background"]

    if normalize:
        # normalize by row (true class) to get per-class recall
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        cm_disp = cm / row_sums
    else:
        cm_disp = cm

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_disp, interpolation="nearest")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # ticks & labels
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)

    # annotate cells (optional, can comment out if cluttered)
    fmt = ".2f" if normalize else "d"
    thresh = cm_disp.max() / 2.0
    for i in range(cm_disp.shape[0]):
        for j in range(cm_disp.shape[1]):
            ax.text(
                j, i,
                format(cm_disp[i, j], fmt),
                ha="center", va="center",
                color="white" if cm_disp[i, j] > thresh else "black",
                fontsize=8,
            )

    plt.tight_layout()
    plt.show()


# ---------------------------------------
# 3) Plot confusion matrices (VAL & TEST)
# ---------------------------------------
plot_confusion_matrix(
    val_metrics,
    class_names,
    title="Validation Confusion Matrix"
)

plot_confusion_matrix(
    test_metrics,
    class_names,
    title="Test Confusion Matrix"
)


## Grad-CAM visualization for YOLO

In [ ]:
# ===============================================================
# Grad-CAM visualization for YOLO (SimCLR-pretrained) on 1 test image
# ===============================================================
from pathlib import Path
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"

# -------- 1) Load best model --------
det_run_name = "yolo12s_simclr_det"
best_pt = WORK / det_run_name / "weights" / "best.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = YOLO(str(best_pt))
det_model = model.model.to(device).eval()

print("Loaded model:", best_pt)

# -------- 2) Pick a test image --------
SPLIT     = WORK / "0_yolo_split"
AUG_SPLIT = WORK / "0_yolo_split_aug"

if SPLIT.exists():
    print("Using original split for Grad-CAM:", SPLIT)
else:
    print("⚠️ Original split not found, using augmented split:", AUG_SPLIT)
    SPLIT = AUG_SPLIT

test_img_dir = SPLIT / "test" / "images"
test_imgs = sorted(
    list(test_img_dir.glob("*.jpg")) +
    list(test_img_dir.glob("*.png")) +
    list(test_img_dir.glob("*.jpeg"))
)
if not test_imgs:
    raise FileNotFoundError(f"No test images found in {test_img_dir}")

img_path = test_imgs[0]      # ekta sample image
print("Using image:", img_path)

# Load original image (for overlay)
orig_bgr = cv2.imread(str(img_path))
orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
H0, W0 = orig_rgb.shape[:2]

# Preprocess for YOLO (640x640)
IMG_SIZE = 640
img_resized = cv2.resize(orig_rgb, (IMG_SIZE, IMG_SIZE))
img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0
img_tensor = img_tensor.unsqueeze(0).to(device)   # [1,3,640,640]

# -------- 3) Register hooks on a backbone layer --------
# We reuse the same idea as before: last backbone before Detect
try:
    from ultralytics.nn.modules.head import Detect
except Exception:
    try:
        from ultralytics.nn.modules import Detect
    except Exception:
        Detect = None

def find_backbone_idx(det_model):
    if hasattr(det_model, "model"):
        modules = det_model.model
    else:
        modules = list(det_model.children())

    if Detect is not None:
        for i, m in enumerate(modules):
            if isinstance(m, Detect):
                return i - 1
    return len(modules) - 2  # fallback

backbone_idx = find_backbone_idx(det_model)
target_module = det_model.model[backbone_idx]
print(f"Grad-CAM hook layer idx {backbone_idx} -> {target_module.__class__.__name__}")

features = {}
gradients = {}

def fwd_hook(module, inp, out):
    # out: [B, C, H, W]
    features["value"] = out

def bwd_hook(module, grad_in, grad_out):
    # grad_out[0]: [B, C, H, W]
    gradients["value"] = grad_out[0]

handle_fwd = target_module.register_forward_hook(fwd_hook)
handle_bwd = target_module.register_full_backward_hook(bwd_hook)

# -------- 4) Forward pass & choose a detection score for Grad-CAM --------
det_model.zero_grad()

with torch.no_grad():
    preds = det_model(img_tensor)  # raw model output

# Ultralytics YOLO: preds[0] is [B, N, 4+1+nc] usually
if isinstance(preds, (list, tuple)):
    pred = preds[0]
else:
    pred = preds

# [1, N, 4+1+nc]
pred = pred[0]  # [N, 4+1+nc]

# Scores per anchor per class: obj * class_prob
obj = pred[:, 4]              # [N]
cls_logits = pred[:, 5:]      # [N, nc]
cls_prob = cls_logits.softmax(dim=1)
cls_max, cls_ids = cls_prob.max(dim=1)  # best class per anchor
scores = obj * cls_max                 # detection score

# pick the highest scoring anchor as Grad-CAM target
best_idx = torch.argmax(scores)
best_score = scores[best_idx]
best_class = cls_ids[best_idx].item()

print(f"Grad-CAM for best detection: score={best_score.item():.3f}, class_id={best_class}")

# Need another forward with grad tracking ON
det_model.zero_grad()
features.clear()
gradients.clear()

img_tensor.requires_grad_(True)
out = det_model(img_tensor)
if isinstance(out, (list, tuple)):
    out = out[0]
out = out[0]  # [N, 4+1+nc]

obj = out[:, 4]
cls_logits = out[:, 5:]
cls_prob = cls_logits.softmax(dim=1)
cls_max, cls_ids = cls_prob.max(dim=1)
scores = obj * cls_max

target_score = scores[best_idx]
target_score.backward()

# -------- 5) Build Grad-CAM heatmap --------
feat = features["value"]        # [1, C, H, W]
grad = gradients["value"]       # [1, C, H, W]

feat = feat[0]   # [C, H, W]
grad = grad[0]   # [C, H, W]

# Global-average-pool gradients over spatial dims
weights = grad.mean(dim=(1, 2))          # [C]
cam = (weights[:, None, None] * feat).sum(dim=0)  # [H, W]

cam = torch.relu(cam)
cam = cam.detach().cpu().numpy()

# Normalize 0–1
cam -= cam.min()
if cam.max() > 0:
    cam /= cam.max()

# Resize to original image size
cam_resized = cv2.resize(cam, (W0, H0))

# Convert to heatmap
heatmap = (cam_resized * 255).astype(np.uint8)
heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

# Overlay: alpha blend
alpha = 0.4
overlay = (alpha * heatmap + (1 - alpha) * orig_rgb).astype(np.uint8)

# Cleanup hooks
handle_fwd.remove()
handle_bwd.remove()

# -------- 6) Show Grad-CAM --------
plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.imshow(orig_rgb)
plt.title("Original image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(heatmap)
plt.title("Grad-CAM heatmap")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title("Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ===============================================================
# Grad-CAM for MULTIPLE test images (YOLO + SimCLR model)
# ===============================================================
from pathlib import Path
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import random

# ----------------- Config -----------------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"

det_run_name = "yolo12s_simclr_det"
best_pt = WORK / det_run_name / "weights" / "best.pt"

IMG_SIZE = 640
N_IMAGES = 6   #  how many images to show (change as you like: 4, 6, 8...)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------- Load model -----------------
print("Loading model:", best_pt)
model = YOLO(str(best_pt))
det_model = model.model.to(device).eval()

# ----------------- Choose test images -----------------
SPLIT     = WORK / "0_yolo_split"
AUG_SPLIT = WORK / "0_yolo_split_aug"

if SPLIT.exists():
    print("Using original split for Grad-CAM:", SPLIT)
else:
    print("⚠️ Original split not found, using augmented split:", AUG_SPLIT)
    SPLIT = AUG_SPLIT

test_img_dir = SPLIT / "test" / "images"
test_imgs = sorted(
    list(test_img_dir.glob("*.jpg")) +
    list(test_img_dir.glob("*.png")) +
    list(test_img_dir.glob("*.jpeg"))
)

if not test_imgs:
    raise FileNotFoundError(f"No test images found in {test_img_dir}")

print(f"Total test images: {len(test_imgs)}")
n_show = min(N_IMAGES, len(test_imgs))
sample_imgs = random.sample(test_imgs, n_show)
print("Selected images for Grad-CAM:")
for p in sample_imgs:
    print("  -", p.name)

# ----------------- Hook setup (same for all images) -----------------
try:
    from ultralytics.nn.modules.head import Detect
except Exception:
    try:
        from ultralytics.nn.modules import Detect
    except Exception:
        Detect = None

def find_backbone_idx(det_model):
    if hasattr(det_model, "model"):
        modules = det_model.model
    else:
        modules = list(det_model.children())
    if Detect is not None:
        for i, m in enumerate(modules):
            if isinstance(m, Detect):
                return i - 1
    return len(modules) - 2  # fallback

backbone_idx = find_backbone_idx(det_model)
target_module = det_model.model[backbone_idx]
print(f"Grad-CAM hook layer idx {backbone_idx} -> {target_module.__class__.__name__}")

features = {}
gradients = {}

def fwd_hook(module, inp, out):
    features["value"] = out

def bwd_hook(module, grad_in, grad_out):
    gradients["value"] = grad_out[0]

handle_fwd = target_module.register_forward_hook(fwd_hook)
handle_bwd = target_module.register_full_backward_hook(bwd_hook)

# ----------------- Grad-CAM function for ONE image -----------------
def grad_cam_overlay_for_path(img_path):
    # Load original image
    orig_bgr = cv2.imread(str(img_path))
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
    H0, W0 = orig_rgb.shape[:2]

    # Resize & normalize
    img_resized = cv2.resize(orig_rgb, (IMG_SIZE, IMG_SIZE))
    img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0
    img_tensor = img_tensor.unsqueeze(0).to(device)   # [1,3,H,W]
    img_tensor.requires_grad_(True)

    # Clear state
    det_model.zero_grad()
    features.clear()
    gradients.clear()

    # Forward
    out = det_model(img_tensor)
    if isinstance(out, (list, tuple)):
        out = out[0]
    out = out[0]   # [N, 4+1+nc]

    # Compute detection scores: obj * max_class_prob
    obj = out[:, 4]               # [N]
    cls_logits = out[:, 5:]       # [N, nc]
    cls_prob = cls_logits.softmax(dim=1)
    cls_max, cls_ids = cls_prob.max(dim=1)
    scores = obj * cls_max        # [N]

    # If everything is zero or NaN, just return original
    if scores.numel() == 0 or torch.all(scores <= 0):
        return orig_rgb, orig_rgb

    best_idx = torch.argmax(scores)
    best_score = scores[best_idx]
    best_class = cls_ids[best_idx].item()

    # Backprop from this score
    target_score = best_score
    target_score.backward()

    # Build CAM
    feat = features["value"][0]     # [C,H,W]
    grad = gradients["value"][0]    # [C,H,W]

    weights = grad.mean(dim=(1, 2))             # [C]
    cam = (weights[:, None, None] * feat).sum(dim=0)  # [H,W]
    cam = torch.relu(cam)
    cam = cam.detach().cpu().numpy()

    # Normalize 0–1
    cam -= cam.min()
    if cam.max() > 0:
        cam /= cam.max()

    # Resize to original
    cam_resized = cv2.resize(cam, (W0, H0))
    heatmap = (cam_resized * 255).astype(np.uint8)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    alpha = 0.4
    overlay = (alpha * heatmap + (1 - alpha) * orig_rgb).astype(np.uint8)

    return orig_rgb, overlay

# ----------------- Generate Grad-CAM overlays -----------------
overlays = []
originals = []

print("\nGenerating Grad-CAM overlays...")
for p in sample_imgs:
    orig, ov = grad_cam_overlay_for_path(p)
    originals.append(orig)
    overlays.append(ov)

# Remove hooks after done
handle_fwd.remove()
handle_bwd.remove()

# ----------------- Plot grid (only overlays, paper-style) -----------------
cols = 3
rows = int(np.ceil(n_show / cols))
plt.figure(figsize=(4 * cols, 4 * rows))

for i, ov in enumerate(overlays):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(ov)
    plt.axis("off")
    plt.title(f"Image {i+1}")

plt.tight_layout()
plt.show()


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# =============================
# Paths
# =============================
BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"
det_run_name = "yolo12s_simclr_det"          # same as in model.train(...)
run_dir = WORK / det_run_name

results_csv = run_dir / "results.csv"
print("Reading:", results_csv)

df = pd.read_csv(results_csv)
print("Columns in results.csv:", df.columns.tolist())

# =============================
# 1) Training + Validation box loss (one plot)
# =============================
plt.figure(figsize=(8, 5))

if "epoch" in df.columns:
    x = df["epoch"]
else:
    x = df.index

plt.plot(x, df["train/box_loss"], label="train/box_loss")

if "val/box_loss" in df.columns:
    plt.plot(x, df["val/box_loss"], label="val/box_loss")

plt.xlabel("Epoch")
plt.ylabel("Box loss")
plt.title("Training / Validation Box Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# =============================
# 2) Validation metrics curves
# =============================
metric_cols = [
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
]

plt.figure(figsize=(8, 5))
for c in metric_cols:
    if c in df.columns:
        plt.plot(x, df[c], label=c.split("/")[-1])

plt.xlabel("Epoch")
plt.ylabel("Metric value")
plt.title("Validation Metrics Over Epochs")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# =============================
# 3) Final epoch bar chart of metrics
# =============================
last = df.iloc[-1]
vals = []
labels = []
for c in metric_cols:
    if c in df.columns:
        labels.append(c.split("/")[-1])
        vals.append(last[c])

if vals:
    plt.figure(figsize=(6, 4))
    plt.bar(labels, vals)
    plt.ylabel("Value")
    plt.title("Final Epoch Metrics")
    plt.grid(axis="y")
    plt.tight_layout()
    plt.show()
else:
    print("No metric columns found in results.csv for bar chart.")


In [ ]:
# ===============================================================
# Core plots for paper using your YOLO+SimCLR model
# ===============================================================
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml
from ultralytics import YOLO

# ----------------- Paths & model -----------------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"
det_run_name = "yolo12s_simclr_det"
run_dir = WORK / det_run_name
results_csv = run_dir / "results.csv"

best_pt = run_dir / "weights" / "best.pt"

DATA_AUG = WORK / "data_simclr_augmented.yaml"
DATA_FALLBACK = WORK / "data_simclr.yaml"
if DATA_AUG.exists():
    DATA = DATA_AUG
    print("Using augmented data yaml:", DATA)
elif DATA_FALLBACK.exists():
    DATA = DATA_FALLBACK
    print("Using fallback data yaml:", DATA)
else:
    raise FileNotFoundError("No data yaml found.")

# Load class names
with open(DATA, "r") as f:
    cfg = yaml.safe_load(f)
class_names = cfg.get("names", [])
nc = len(class_names)

# Load model
device_idx = 0 if torch.cuda.is_available() else None
model = YOLO(str(best_pt))

# A bit nicer default styling for paper figures
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "legend.fontsize": 11,
})


# ===============================================================
# 1) Training + Validation box loss vs epoch
# ===============================================================
df = pd.read_csv(results_csv)
print("Columns in results.csv:", df.columns.tolist())

epochs = df["epoch"] if "epoch" in df.columns else df.index

plt.figure(figsize=(7, 5), dpi=150)
plt.plot(epochs, df["train/box_loss"], label="Train box loss", linewidth=2)
if "val/box_loss" in df.columns:
    plt.plot(epochs, df["val/box_loss"], label="Val box loss", linewidth=2)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Box Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


# ===============================================================
# 2) Validation metrics vs epoch (precision, recall, mAP)
# ===============================================================
metric_cols = [
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
]

plt.figure(figsize=(7, 5), dpi=150)
for c in metric_cols:
    if c in df.columns:
        label = c.split("/")[-1].replace("(B)", "")
        plt.plot(epochs, df[c], label=label, linewidth=2)

plt.xlabel("Epoch")
plt.ylabel("Metric value")
plt.title("Validation Metrics over Epochs")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


# ===============================================================
# 3) Confusion matrix on TEST split (normalized)
# ===============================================================
def plot_confusion_matrix(metrics, class_names, title="Test Confusion Matrix (normalized)"):
    cm = np.array(metrics.confusion_matrix.matrix, dtype=float)

    # keep background row/col if present
    labels = class_names + ["background"]

    # normalize per true class (rows)
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_norm = cm / row_sums

    fig, ax = plt.subplots(figsize=(7, 6), dpi=150)
    im = ax.imshow(cm_norm, interpolation="nearest")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)

    # annotate cells (optional)
    fmt = ".2f"
    thresh = cm_norm.max() / 2.0
    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            ax.text(
                j, i, format(cm_norm[i, j], fmt),
                ha="center", va="center",
                color="white" if cm_norm[i, j] > thresh else "black",
                fontsize=8,
            )
    plt.tight_layout()
    plt.show()


print("\nRunning val() on TEST split to get confusion matrix...")
test_metrics = model.val(
    data=str(DATA),
    imgsz=640,
    split="test",
    batch=8,
    device=device_idx,
    verbose=False,
)

plot_confusion_matrix(test_metrics, class_names)


# ===============================================================
# 4) Per-class AP bar chart (mAP@0.5:0.95 on TEST)
# ===============================================================
# Ultralytics: metrics.box.maps is per-class AP (IoU 0.5:0.95)
try:
    per_class_ap = np.array(test_metrics.box.maps)
    if per_class_ap.size == 0:
        raise ValueError("Empty AP array from metrics.box.maps")

    # truncate / pad names to match AP length
    n_cls_ap = len(per_class_ap)
    labels = class_names[:n_cls_ap]

    plt.figure(figsize=(max(7, 0.6 * n_cls_ap + 2), 5), dpi=150)
    x = np.arange(n_cls_ap)
    plt.bar(x, per_class_ap)
    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylabel("AP (IoU 0.5:0.95)")
    plt.title("Per-class Average Precision on Test Set")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print("Per-class AP bar chart not available:", e)


In [ ]:
# ===============================================================
# Confidence distribution & confidence curve (CDF) on TEST set
# ===============================================================
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
import cv2

# ---------- Paths ----------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"

# Original + augmented splits
SPLIT     = WORK / "0_yolo_split"
AUG_SPLIT = WORK / "0_yolo_split_aug"

# Choose test image dir
if SPLIT.exists():
    print("Using original split for test images:", SPLIT)
else:
    print("⚠️ Original split not found, using augmented split:", AUG_SPLIT)
    SPLIT = AUG_SPLIT

test_img_dir = SPLIT / "test" / "images"
if not test_img_dir.exists():
    raise FileNotFoundError(f"Test image dir not found: {test_img_dir}")

test_imgs = sorted(
    list(test_img_dir.glob("*.jpg")) +
    list(test_img_dir.glob("*.png")) +
    list(test_img_dir.glob("*.jpeg"))
)
print(f"Found {len(test_imgs)} test images.")

# ---------- Load best model ----------
det_run_name = "yolo12s_simclr_det"
best_pt = WORK / det_run_name / "weights" / "best.pt"

try:
    best_model
except NameError:
    print("Loading best model from:", best_pt)
    best_model = YOLO(str(best_pt))

device_idx = 0 if torch.cuda.is_available() else None

# ---------- Run predictions & collect confidences ----------
all_confs = []

BATCH_SIZE = 16  # adjust if memory issues

print("\nCollecting confidences from test predictions...")
for i in range(0, len(test_imgs), BATCH_SIZE):
    batch_paths = [str(p) for p in test_imgs[i : i + BATCH_SIZE]]
    results = best_model.predict(
        source=batch_paths,
        imgsz=640,
        conf=0.001,          # very low threshold to see full distribution
        device=device_idx,
        verbose=False,
    )

    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        confs = r.boxes.conf.cpu().numpy()
        all_confs.append(confs)

if not all_confs:
    raise RuntimeError("No predictions found on test set to build confidence curve.")

all_confs = np.concatenate(all_confs, axis=0)
print(f"Collected {all_confs.shape[0]} box confidences.")

# ===============================================================
# 1) Confidence histogram
# ===============================================================
plt.figure(figsize=(7, 5), dpi=150)
bins = np.linspace(0.0, 1.0, 21)  # 0.0–1.0 in steps of 0.05
plt.hist(all_confs, bins=bins, edgecolor="black", alpha=0.7)
plt.xlabel("Confidence score")
plt.ylabel("Number of detections")
plt.title("Confidence Distribution of YOLO Detections (Test Set)")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# ===============================================================
# 2) Confidence CDF (Confidence curve)
# ===============================================================
sorted_confs = np.sort(all_confs)
n = len(sorted_confs)
cdf = np.arange(1, n + 1) / n  # fraction of boxes with conf ≤ x

plt.figure(figsize=(7, 5), dpi=150)
plt.plot(sorted_confs, cdf, linewidth=2)
plt.xlabel("Confidence score")
plt.ylabel("Cumulative fraction of detections")
plt.title("Confidence Curve (CDF) of Detection Scores on Test Set")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## SAVE BEST MODEL FOR DEPLOYMENT

In [ ]:
# Save / Export models for web or app usage
from ultralytics import YOLO
from pathlib import Path
import shutil
import torch

BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"

# Paths from your training
det_run_name = "yolo12s_simclr_det"
best_pt = WORK / det_run_name / "weights" / "best.pt"
simclr_backbone_pt = WORK / "simclr_backbone_with_hooks_yolov12s.pt"

print("Detection best.pt:", best_pt)
print("SimCLR backbone  :", simclr_backbone_pt)

# 1) Create export directory
EXPORT_DIR = WORK / "exported_models"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print("\nExport directory:", EXPORT_DIR)

# 2) Copy raw PyTorch checkpoints into export folder
det_export_pt = EXPORT_DIR / "yolo12s_simclr_det_best.pt"
backbone_export_pt = EXPORT_DIR / "simclr_backbone_yolov12s.pt"

shutil.copy2(best_pt, det_export_pt)
shutil.copy2(simclr_backbone_pt, backbone_export_pt)

print("\nCopied:")
print(" - Detection model ->", det_export_pt)
print(" - SimCLR backbone  ->", backbone_export_pt)

# 3) Export detection model to ONNX (for deployment)
print("\nLoading detection model for ONNX export...")
model = YOLO(str(det_export_pt))

device = 0 if torch.cuda.is_available() else "cpu"

onnx_export_dir = EXPORT_DIR / "onnx_export"
onnx_export_dir.mkdir(parents=True, exist_ok=True)

print("Exporting to ONNX …")
onnx_path = model.export(
    format="onnx",
    imgsz=640,           # match your training/inference size
    opset=12,            # common opset
    dynamic=False,       # set True if you want dynamic shapes
    simplify=True,       # try to simplify the ONNX graph
    project=str(onnx_export_dir),
    name="yolo12s_simclr_onnx",
    exist_ok=True,
)

print("\n✓ Export complete.")
print("PyTorch detection checkpoint   :", det_export_pt)
print("PyTorch SimCLR backbone        :", backbone_export_pt)
print("ONNX model (for deployment)    :", onnx_path)


In [28]:
import pickle
from pathlib import Path
from ultralytics import YOLO
import torch

BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"

det_run_name = "yolo12s_simclr_det"
best_pt = WORK / det_run_name / "weights" / "best.pt"
print("Best checkpoint:", best_pt)

# Load best_model if not already in memory
try:
    best_model
except NameError:
    best_model = YOLO(str(best_pt))

# Where to save the pkl
EXPORT_DIR = WORK / "exported_models"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pkl_path = EXPORT_DIR / "yolo12s_simclr_det_best.pkl"

# Package what we need into a dict
model_package = {
    "model_type": "ultralytics_yolo12s_simclr",
    "cfg_or_weights": "yolo12s.pt",          # base config/weights file used
    "state_dict": best_model.model.state_dict(),
    "class_names": best_model.names,
}

with open(pkl_path, "wb") as f:
    pickle.dump(model_package, f)

print("✓ Saved YOLO model package to:", pkl_path)


Best checkpoint: /kaggle/working/palm_simclr_ssl/yolo12s_simclr_det/weights/best.pt
✓ Saved YOLO model package to: /kaggle/working/palm_simclr_ssl/exported_models/yolo12s_simclr_det_best.pkl


In [29]:
from pathlib import Path

BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"
MODEL_PKL_PATH = WORK / "exported_models" / "yolo12s_simclr_det_best.pkl"

print("Model PKL:", MODEL_PKL_PATH)


Model PKL: /kaggle/working/palm_simclr_ssl/exported_models/yolo12s_simclr_det_best.pkl


In [30]:
from pathlib import Path
import shutil

BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"

det_run_name = "yolo12s_simclr_det"
best_src = WORK / det_run_name / "weights" / "best.pt"

# Where we'll store files for deployment
EXPORT_DIR = WORK / "deploy_artifacts"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Standard name for deployment
det_pt = EXPORT_DIR / "yolo_palm_best.pt"
shutil.copy2(best_src, det_pt)

print("✓ Detection model copied for deployment:")
print("  ", det_pt)


✓ Detection model copied for deployment:
   /kaggle/working/palm_simclr_ssl/deploy_artifacts/yolo_palm_best.pt


### Save the .pt file into the Local PC

In [31]:
from pathlib import Path
from IPython.display import FileLink
import os

BASE = Path("/kaggle/working")
WORK = BASE / "palm_simclr_ssl"
det_pt = WORK / "deploy_artifacts" / "yolo_palm_best.pt"

# Check that the file really exists
print("Exists?", det_pt.exists(), " -> ", det_pt)

# Make sure notebook root is /kaggle/working
os.chdir(BASE)

# Use a relative path for the link
rel_path = det_pt.relative_to(BASE)
FileLink(str(rel_path))


Exists? True  ->  /kaggle/working/palm_simclr_ssl/deploy_artifacts/yolo_palm_best.pt


/kaggle/working/palm_simclr_ssl/deploy_artifacts/yolo_palm_best.pt

In [ ]:
 plt.figure(figsize=(8, 5), dpi=150)
 for col in ["train/box_loss", "train/cls_loss", "train/dfl_loss"]:
     if col in df.columns:
         plt.plot(epochs, df[col], label=col)
 plt.xlabel("Epoch")
 plt.ylabel("Loss")
 plt.title("Training losses (box / cls / dfl)")
 plt.legend()
 plt.grid(True, linestyle="--", alpha=0.4)
 plt.tight_layout()
 plt.show()
